<a href="https://colab.research.google.com/github/DZT711/OllamaHostingOnCloudUsingNgrok/blob/main/ollamaNgrokDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Download & setup ollama/ngrok sever require ngrok auth token

In [ ]:
# ==============================================================================
# 1. CÀI ĐẶT HỆ THỐNG & FIX LỖI GPU (lspci)
# ==============================================================================
import os
import subprocess
import time

print("1. Đang tối ưu hệ thống cho GPU...")

# Cài đặt zstd và pciutils (để fix lỗi 'Unable to detect GPU' trong image_1f82be.png)
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw

# Kiểm tra xem Ollama đã được cài đặt chưa
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    print("📥 Đang cài đặt Ollama... Vui lòng đợi...")
    !curl -fsSL https://ollama.com/install.sh | sh
    print("✅ Cài đặt Ollama hoàn tất!")
else:
    print("✅ Ollama đã được cài đặt, bỏ qua bước cài đặt.")

# ==============================================================================
# 2. KHỞI CHẠY MÁY CHỦ OLLAMA
# ==============================================================================
print("\n2. Đang khởi chạy máy chủ Ollama ngầm...")
!pkill ollama
!pkill ngrok

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

time.sleep(5) # Chờ server ổn định

# ==============================================================================
# 3. QUẢN LÝ MODEL (CHỈ TẢI KHI CẦN THIẾT)
# ==============================================================================
print("\n3. --- DANH SÁCH MODEL HIỆN CÓ ---")
!ollama list
print("----------------------------------")

# Tên model bạn muốn sử dụng
TARGET_MODEL = "llama3"

# Kiểm tra xem model đã có trong máy chưa
check_model = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if TARGET_MODEL in check_model.stdout:
    print(f"✅ Model '{TARGET_MODEL}' đã sẵn sàng, không cần tải lại.")
else:
    print(f"📥 Model '{TARGET_MODEL}' chưa có. Đang tiến hành tải...")
    !ollama pull {TARGET_MODEL}

# ==============================================================================
# 4. THIẾT LẬP NGROK URL
# ==============================================================================
print("\n4. Đang thiết lập đường hầm Ngrok...")
!pip install pyngrok --quiet
from pyngrok import ngrok

# Thay mã Token của bạn vào đây
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

if NGROK_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    print("❌ LỖI: Vui lòng điền Ngrok Token!")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    try:
        public_url = ngrok.connect(11434)
        print("\n" + "="*60)
        print(f"🎉 MÁY CHỦ OLLAMA ĐÃ SẴN SÀNG!")
        print(f"🔗 Base URL của bạn: {public_url.public_url}")
        print(f"🤖 Đang chạy model: {TARGET_MODEL}")
        print("="*60)
    except Exception as e:
        print("\n❌ Lỗi Ngrok:", e)

2. Download additional python libaries

In [ ]:
!pip -q install playwright
!playwright install chromium

3. Agent point column(not accurated due to uncrawable to all model information )

In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - FULL SRC
# STABLE FINAL PATCH
#
# FEATURES:
# ✅ Ollama Library
# ✅ HuggingFace GGUF Crawl
# ✅ LM Studio Crawl
# ✅ Provider Sort
# ✅ Context Detection
# ✅ Size Detection
# ✅ Quant Detection
# ✅ Compatibility Score
# ✅ Agent Capability Score
# ✅ Tool YES/NO
# ✅ Page System
# ✅ Delete by ID
# ✅ Delete by Name
# ✅ Info with URL
# ✅ Realtime Pull Logs
# ✅ Ngrok
# ==============================================================================

import os
import re
import time
import json
import requests
import subprocess

from urllib.parse import quote, unquote

import ipywidgets as widgets

from IPython.display import display, clear_output

# ==============================================================================
# CONFIG
# ==============================================================================

OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_PORT = "11434"

MODELS_PER_PAGE = 35
CURRENT_PAGE = 0

MAX_LIBRARY_PAGES = 25

HTTP_TIMEOUT = 30

HF_MAX_MODELS = 1000

LMSTUDIO_PUBLISHERS = [

    "lmstudio-community",
    "bartowski",
    "QuantFactory",
    "unsloth",

]

# ==============================================================================
# STATE
# ==============================================================================

STATE = {

    "installed": [],

    "library": [],

    "details": {},

}

# ==============================================================================
# MODEL DB
# ==============================================================================

MODEL_DB = {

    "llama3.1": {
        "s": "4.7GB",
        "c": "128K",
        "n": "General AI / multilingual / strong Vietnamese."
    },

    "llama3.2": {
        "s": "2GB",
        "c": "128K",
        "n": "Small lightweight model."
    },

    "deepseek-r1": {
        "s": "4.7GB+",
        "c": "128K",
        "n": "Strong reasoning & coding."
    },

    "qwen2.5-coder": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Excellent coding assistant."
    },

    "phi4": {
        "s": "8GB",
        "c": "128K",
        "n": "Strong logic & math."
    },

}

# ==============================================================================
# CMD
# ==============================================================================

def run_cmd(args, timeout=120):

    try:

        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return (
            p.returncode,
            p.stdout.strip(),
            p.stderr.strip()
        )

    except Exception as e:

        return (
            1,
            "",
            str(e)
        )

# ==============================================================================
# CURL
# ==============================================================================

def curl_get(url):

    code, out, err = run_cmd([

        "curl",
        "-L",
        "-sS",
        "--max-time",
        str(HTTP_TIMEOUT),
        url

    ])

    if code != 0:
        return ""

    return out

# ==============================================================================
# PROVIDER PRIORITY
# ==============================================================================

def get_provider_priority(name):

    if not name.startswith("hf.co/") and not name.startswith("lmstudio/"):
        return 0

    if name.startswith("lmstudio/"):
        return 1

    if name.startswith("hf.co/"):
        return 2

    return 999

# ==============================================================================
# AGENT SCORE
# ==============================================================================

def compute_agent_score(name, description="", context="", quant=""):

    text = (
        f"{name} {description} {context} {quant}"
    ).lower()

    score = 0
    reasons = []

    keywords = {

        "tool": 20,
        "function calling": 25,
        "agent": 20,
        "json": 10,
        "coder": 20,
        "reasoning": 20,
        "r1": 15,
        "instruct": 10,
        "code": 10,
        "tools": 20,

    }

    for k, pts in keywords.items():

        if k in text:

            score += pts
            reasons.append(k)

    # context score
    ctx_num = 0

    ctx_match = re.search(
        r'([0-9]+)\s*K',
        context,
        flags=re.I
    )

    if ctx_match:

        try:

            ctx_num = int(
                ctx_match.group(1)
            )

        except:
            pass

    if ctx_num >= 128:
        score += 20

    elif ctx_num >= 64:
        score += 10

    elif ctx_num >= 32:
        score += 5

    if score >= 70:
        tool = "YES"

    elif score >= 40:
        tool = "PARTIAL"

    else:
        tool = "UNVERIFIED"

    compat = score >= 45

    return {

        "score": min(score, 100),

        "tool": tool,

        "compat": compat,

        "reasons": reasons

    }

# ==============================================================================
# RESTART SERVER
# ==============================================================================

def restart_ollama_server():

    print("🔄 Starting Ollama...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    os.environ["OLLAMA_HOST"] = OLLAMA_HOST

    log_file = open("ollama.log", "w")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=log_file
    )

    time.sleep(6)

    print("✅ Ollama Ready!")

# ==============================================================================
# STOP SERVER
# ==============================================================================

def stop_ollama_server():

    print("\n🛑 Stopping Ollama...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    print("✅ Ollama Stopped!")

# ==============================================================================
# NGROK
# ==============================================================================

def run_ngrok():

    print("\n🚀 Starting Ngrok...")

    run_cmd([
        "bash",
        "-lc",
        f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"
    ])

    time.sleep(3)

    try:

        response = requests.get(
            "http://localhost:4040/api/tunnels",
            timeout=5
        )

        tunnels = response.json().get("tunnels", [])

        if tunnels:

            url = tunnels[0]["public_url"]

            print("\n✅ NGROK ONLINE:")
            print(url)

        else:

            print("⚠️ No Tunnel Found")

    except:

        print("⚠️ Ngrok Error")

# ==============================================================================
# STOP NGROK
# ==============================================================================

def stop_ngrok():

    print("\n🛑 Stopping Ngrok...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ngrok || true"
    ])

    print("✅ Ngrok Stopped!")

# ==============================================================================
# INSTALLED MODELS
# ==============================================================================

def get_installed_models():

    code, out, err = run_cmd(
        ["ollama", "list"],
        timeout=20
    )

    if code != 0:
        return []

    lines = out.splitlines()

    if len(lines) <= 1:
        return []

    models = []

    for line in lines[1:]:

        parts = line.split()

        if not parts:
            continue

        name = parts[0].split(":")[0]

        models.append(name)

    return models

# ==============================================================================
# FETCH OLLAMA
# ==============================================================================

def fetch_library_models():

    models = []

    for page in range(1, MAX_LIBRARY_PAGES + 1):

        try:

            if page == 1:
                url = "https://ollama.com/library"
            else:
                url = f"https://ollama.com/library?page={page}"

            html = curl_get(url)

            if not html:
                continue

            found = re.findall(
                r'href="/library/([^"]+)"',
                html
            )

            for item in found:

                item = unquote(item)

                if "/" in item:
                    continue

                if item not in models:
                    models.append(item)

        except:
            pass

    return models

# ==============================================================================
# FETCH OLLAMA DETAILS
# ==============================================================================

def fetch_ollama_detail(slug):

    result = {

        "size": "---",
        "context": "---",
        "description": "",
        "url": f"https://ollama.com/library/{slug}",
        "quant": "---",

    }

    try:

        html = curl_get(
            f"https://ollama.com/library/{quote(slug)}"
        )

        if html:

            desc = re.search(
                r'<meta name="description" content="([^"]+)"',
                html
            )

            if desc:
                result["description"] = desc.group(1)

        tags_html = curl_get(
            f"https://ollama.com/library/{quote(slug)}/tags"
        )

        if tags_html:

            text = re.sub(
                r"<[^>]+>",
                " ",
                tags_html
            )

            size_match = re.search(
                r'([0-9.]+\s?(?:GB|MB))',
                text
            )

            ctx_match = re.search(
                r'([0-9.]+\s?[KMG]?)\s*context',
                text,
                flags=re.I
            )

            quant_match = re.search(
                r'(Q[0-9][^ ]*)',
                text,
                flags=re.I
            )

            if size_match:
                result["size"] = size_match.group(1)

            if ctx_match:
                result["context"] = ctx_match.group(1)

            if quant_match:
                result["quant"] = quant_match.group(1)

    except:
        pass

    return result

# ==============================================================================
# FETCH HF
# ==============================================================================

def fetch_huggingface_models():

    models = []

    try:

        url = (
            "https://huggingface.co/api/models"
            "?search=GGUF"
            # "&limit=1000"
            "&full=true"
        )

        response = requests.get(
            url,
            timeout=40
        )

        data = response.json()

        for item in data:

            model_id = item.get("id", "")

            if not model_id:
                continue

            siblings = item.get("siblings", [])

            gguf_files = []

            total_size = 0

            for s in siblings:

                fname = s.get(
                    "rfilename",
                    ""
                ).lower()

                if fname.endswith(".gguf"):

                    gguf_files.append(fname)

                    size = s.get("size")

                    if isinstance(size, int):
                        total_size += size

            if not gguf_files:
                continue

            quant = "---"

            for q in [

                "q2_k",
                "q3_k",
                "q4_0",
                "q4_k_m",
                "q5_k_m",
                "q6_k",
                "q8_0"

            ]:

                if any(q in x for x in gguf_files):

                    quant = q.upper()
                    break

            size_text = "---"

            if total_size > 0:

                gb = total_size / (1024 ** 3)

                if gb >= 1:
                    size_text = f"{gb:.1f}GB"

                else:

                    mb = total_size / (1024 ** 2)
                    size_text = f"{mb:.0f}MB"

            context = "---"

            tags = item.get("tags", [])

            for t in tags:

                if "context" in t.lower():
                    context = t

            desc = item.get("description", "")

            models.append({

                "name": f"hf.co/{model_id}",

                "quant": quant,

                "size": size_text,

                "context": context,

                "description": desc,

                "url": f"https://huggingface.co/{model_id}"

            })

            if len(models) >= HF_MAX_MODELS:
                break

    except Exception as e:

        print(f"HF ERROR: {e}")

    return models

# ==============================================================================
# FETCH LM STUDIO
# ==============================================================================

def fetch_lmstudio_models():

    models = []

    try:

        for publisher in LMSTUDIO_PUBLISHERS:

            url = (
                f"https://huggingface.co/api/models"
                f"?author={publisher}"
                # f"&limit=1000"
                f"&full=true"
            )

            response = requests.get(
                url,
                timeout=40
            )

            data = response.json()

            for item in data:

                model_id = item.get("id", "")

                if not model_id:
                    continue

                siblings = item.get("siblings", [])

                gguf_files = []

                total_size = 0

                for s in siblings:

                    fname = s.get(
                        "rfilename",
                        ""
                    ).lower()

                    if fname.endswith(".gguf"):

                        gguf_files.append(fname)

                        size = s.get("size")

                        if isinstance(size, int):
                            total_size += size

                if not gguf_files:
                    continue

                quant = "---"

                for q in [

                    "q2_k",
                    "q3_k",
                    "q4_0",
                    "q4_k_m",
                    "q5_k_m",
                    "q6_k",
                    "q8_0"

                ]:

                    if any(q in x for x in gguf_files):

                        quant = q.upper()
                        break

                size_text = "---"

                if total_size > 0:

                    gb = total_size / (1024 ** 3)

                    if gb >= 1:
                        size_text = f"{gb:.1f}GB"

                    else:
                        mb = total_size / (1024 ** 2)
                        size_text = f"{mb:.0f}MB"

                desc = item.get("description", "")

                models.append({

                    "name": f"lmstudio/{model_id}",

                    "quant": quant,

                    "size": size_text,

                    "context": "---",

                    "description": desc,

                    "url": f"https://huggingface.co/{model_id}"

                })

    except Exception as e:

        print(f"LM ERROR: {e}")

    return models

# ==============================================================================
# REFRESH
# ==============================================================================

def refresh_all_data():

    STATE["installed"] = get_installed_models()

    ollama_models = fetch_library_models()

    hf_models = fetch_huggingface_models()

    lm_models = fetch_lmstudio_models()

    merged = []

    merged.extend(ollama_models)

    merged.extend([
        x["name"]
        for x in hf_models
    ])

    merged.extend([
        x["name"]
        for x in lm_models
    ])

    merged = list(dict.fromkeys(merged))

    merged = sorted(
        merged,
        key=lambda x: (
            get_provider_priority(x),
            x.lower()
        )
    )

    STATE["library"] = merged

    STATE["details"] = {}

    # OLLAMA
    for model in ollama_models:

        detail = fetch_ollama_detail(model)

        score_data = compute_agent_score(

            model,

            detail["description"],

            detail["context"],

            detail["quant"]

        )

        STATE["details"][model] = {

            **detail,

            **score_data,

        }

    # HF
    for item in hf_models:

        score_data = compute_agent_score(

            item["name"],

            item["description"],

            item["context"],

            item["quant"]

        )

        STATE["details"][item["name"]] = {

            **item,

            **score_data,

        }

    # LM
    for item in lm_models:

        score_data = compute_agent_score(

            item["name"],

            item["description"],

            item["context"],

            item["quant"]

        )

        STATE["details"][item["name"]] = {

            **item,

            **score_data,

        }

# ==============================================================================
# PULL
# ==============================================================================

def pull_model(model_name):

    print("\n" + "═" * 100)

    print(f"📥 DOWNLOADING: {model_name}")

    print("═" * 100)

    try:

        process = subprocess.Popen(

            ["ollama", "pull", model_name],

            stdout=subprocess.PIPE,

            stderr=subprocess.STDOUT,

            text=True,

            bufsize=1

        )

        last_line = ""

        while True:

            line = process.stdout.readline()

            if not line and process.poll() is not None:
                break

            if not line:
                continue

            line = line.strip()

            if not line:
                continue

            pretty = line

            pretty = pretty.replace(
                "pulling manifest",
                "📦 pulling manifest"
            )

            pretty = pretty.replace(
                "verifying sha256 digest",
                "🔐 verifying digest"
            )

            pretty = pretty.replace(
                "writing manifest",
                "💾 writing manifest"
            )

            pretty = pretty.replace(
                "success",
                "✅ success"
            )

            if "%" in pretty:
                pretty = "⬇️ " + pretty

            if pretty != last_line:

                print(pretty)

                last_line = pretty

        rc = process.poll()

        print("\n" + "═" * 100)

        if rc == 0:
            print("✅ DOWNLOAD SUCCESS")
        else:
            print("⚠️ DOWNLOAD FAILED")

    except Exception as e:

        print(f"❌ ERROR: {e}")

# ==============================================================================
# DELETE
# ==============================================================================

def delete_model(model_name):

    print(f"\n🗑️ Removing: {model_name}")

    run_cmd([
        "ollama",
        "rm",
        model_name
    ])

    print("✅ Removed!")

# ==============================================================================
# INFO
# ==============================================================================

def show_model_info(model_name):

    detail = STATE["details"].get(
        model_name,
        {}
    )

    model_id = None

    try:

        model_id = (
            STATE["library"].index(model_name)
            + 1
        )

    except:
        pass

    print("\n" + "═" * 100)

    print(f"MODEL: {model_name}")

    if model_id:
        print(f"ID: {model_id}")

    print("═" * 100)

    print(f"SIZE: {detail.get('size', '---')}")
    print(f"CONTEXT: {detail.get('context', '---')}")
    print(f"QUANT: {detail.get('quant', '---')}")
    print(f"TOOL: {detail.get('tool', '---')}")
    print(f"AGENT SCORE: {detail.get('score', '---')}")
    print(f"COMPATIBLE: {detail.get('compat', '---')}")

    desc = detail.get("description", "---")

    print(f"DESC: {desc}")

    print(f"URL: {detail.get('url', '---')}")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================

def process_command(choice):

    global CURRENT_PAGE

    low = choice.lower()

    # PAGE
    if low.startswith("page "):

        try:

            page_num = int(
                low.split()[1]
            )

            max_page = max(
                1,
                (len(STATE["library"]) - 1)
                // MODELS_PER_PAGE + 1
            )

            if 1 <= page_num <= max_page:

                CURRENT_PAGE = page_num - 1

                render_catalog()

        except:

            print("⚠️ PAGE ERROR")

        return

    # DELETE NAME
    if low.startswith('delete "'):

        try:

            model_name = re.findall(

                r'delete\s+"(.+)"',

                choice,

                flags=re.I

            )[0]

            delete_model(model_name)

            refresh_all_data()

            render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    # DELETE ID
    if low.startswith("delete "):

        try:

            delete_id = int(
                choice.split()[1]
            )

            idx = delete_id - 101

            if 0 <= idx < len(STATE["installed"]):

                target = STATE["installed"][idx]

                delete_model(target)

                refresh_all_data()

                render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    # INFO
    if low.startswith("info "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            idx = int(arg) - 1

            if 0 <= idx < len(STATE["library"]):

                show_model_info(
                    STATE["library"][idx]
                )

        else:

            show_model_info(arg)

        return

    # REFRESH
    if low == "refresh":

        refresh_all_data()

        render_catalog()

        return

    # STOP NGROK
    if low == "stop ngrok":

        stop_ngrok()

        return

    # STOP SERVER
    if low == "stop server":

        stop_ollama_server()

        return

    # NGROK
    if low == "0":

        run_ngrok()

        return

    # PULL BY ID
    if choice.isdigit():

        idx = int(choice) - 1

        if 0 <= idx < len(STATE["library"]):

            target = STATE["library"][idx]

            pull_model(target)

            refresh_all_data()

            render_catalog()

        return

    # MANUAL PULL
    pull_model(choice)

    refresh_all_data()

    render_catalog()

# ==============================================================================
# RENDER
# ==============================================================================

def render_catalog():

    with OUTPUT:

        clear_output(wait=True)

        total_models = len(
            STATE["library"]
        )

        max_page = max(
            1,
            (total_models - 1)
            // MODELS_PER_PAGE + 1
        )

        start = CURRENT_PAGE * MODELS_PER_PAGE

        end = start + MODELS_PER_PAGE

        page_models = STATE["library"][start:end]

        print("═" * 180)

        print(
            "ID   | SRC | STATUS         | MODEL                              | SIZE     | CTX      | QUANT    | TOOL       | SCORE | DESCRIPTION"
        )

        print("─" * 180)

        installed = set(
            STATE["installed"]
        )

        for offset, slug in enumerate(
            page_models
        ):

            i = start + offset + 1

            detail = STATE["details"].get(
                slug,
                {}
            )

            compat = detail.get("compat")

            quant = detail.get("quant", "---")

            if slug.startswith("hf.co/"):

                if compat is True:
                    source = "🤗🟢"

                elif compat is False:
                    source = "🤗🔴"

                else:
                    source = "🤗"

            elif slug.startswith("lmstudio/"):

                if compat is True:
                    source = "🧠🟢"

                elif compat is False:
                    source = "🧠🔴"

                else:
                    source = "🧠"

            else:

                source = "🦙"

            if slug in installed:
                status = "✅ INSTALLED"
            else:
                status = "⚪ EMPTY"

            fallback = MODEL_DB.get(
                slug.lower(),
                {}
            )

            size = detail.get(
                "size",
                fallback.get("s", "---")
            )

            context = detail.get(
                "context",
                fallback.get("c", "---")
            )

            desc = detail.get(
                "description",
                fallback.get("n", "")
            )

            tool = detail.get(
                "tool",
                "---"
            )

            score = detail.get(
                "score",
                "---"
            )

            if len(desc) > 45:
                desc = desc[:45] + "..."

            display_slug = slug[:34]

            print(

                f"{str(i):<4} | "

                f"{source:<4} | "

                f"{status:<14} | "

                f"{display_slug:<34} | "

                f"{size:<8} | "

                f"{context:<8} | "

                f"{quant:<8} | "

                f"{tool:<10} | "

                f"{str(score):<5} | "

                f"{desc}"

            )

        print("\n📂 INSTALLED MODELS:")

        for i, name in enumerate(

            STATE["installed"],

            101

        ):

            print(
                f"{i:<4} | 🗑️ {name}"
            )

        print("\n" + "═" * 180)

        ollama_count = len([
            x for x in STATE["library"]
            if not x.startswith("hf.co/")
            and not x.startswith("lmstudio/")
        ])

        lmstudio_count = len([
            x for x in STATE["library"]
            if x.startswith("lmstudio/")
        ])

        hf_count = len([
            x for x in STATE["library"]
            if x.startswith("hf.co/")
        ])

        print(
            f"🦙 Ollama: {ollama_count} | "
            f"🧠 LM Studio: {lmstudio_count} | "
            f"🤗 HF: {hf_count}"
        )

        print(
            f"\n📄 PAGE: "
            f"{CURRENT_PAGE + 1}/{max_page}"
        )

        print("\nCOMMANDS:")

        print("1                    -> pull model")
        print("delete 101           -> delete by id")
        print('delete "llama3"      -> delete by name')
        print("info 1               -> model info")
        print("page 5               -> jump page")
        print("refresh              -> refresh")
        print("stop ngrok           -> stop ngrok")
        print("stop server          -> stop ollama")
        print("0                    -> start ngrok")

# ==============================================================================
# SUBMIT
# ==============================================================================

def on_submit(_=None):

    cmd = INPUT_BOX.value.strip()

    INPUT_BOX.value = ""

    if not cmd:
        return

    with OUTPUT:

        print(f"\n👉 {cmd}")

    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================

restart_ollama_server()

refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================

TITLE = widgets.HTML("""
<h2>🚀 Ollama Colab Manager</h2>
""")

HELP = widgets.HTML("""
<b>
Commands:
1 | delete 101 | info 1 | stop ngrok | stop server
</b>
""")

INPUT_BOX = widgets.Text(

    placeholder="Nhập lệnh...",

    description=">>>",

    layout=widgets.Layout(width="90%")

)

SEND_BTN = widgets.Button(

    description="Gửi",

    button_style="success"

)

REFRESH_BTN = widgets.Button(

    description="Refresh",

    button_style="info"

)

NGROK_BTN = widgets.Button(

    description="Start Ngrok",

    button_style="warning"

)

STOP_NGROK_BTN = widgets.Button(

    description="Stop Ngrok",

    button_style="danger"

)

STOP_SERVER_BTN = widgets.Button(

    description="Stop Server",

    button_style="danger"

)

OUTPUT = widgets.Output()

# ==============================================================================
# EVENTS
# ==============================================================================

SEND_BTN.on_click(on_submit)

INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(

    lambda _: (
        refresh_all_data(),
        render_catalog()
    )

)

NGROK_BTN.on_click(
    lambda _: run_ngrok()
)

STOP_NGROK_BTN.on_click(
    lambda _: stop_ngrok()
)

STOP_SERVER_BTN.on_click(
    lambda _: stop_ollama_server()
)

# ==============================================================================
# BUTTONS
# ==============================================================================

BUTTONS = widgets.HBox([

    SEND_BTN,

    REFRESH_BTN,

    NGROK_BTN,

    STOP_NGROK_BTN,

    STOP_SERVER_BTN

])

# ==============================================================================
# MAIN UI
# ==============================================================================

UI = widgets.VBox([

    TITLE,

    OUTPUT,

    HELP,

    INPUT_BOX,

    BUTTONS

])

display(UI)

render_catalog()

🔄 Starting Ollama...
✅ Ollama Ready!



════════════════════════════════════════════════════════════════════════════════════════════════════
MODEL: llama3
ID: 100
════════════════════════════════════════════════════════════════════════════════════════════════════
SIZE: 4.7GB
CONTEXT: 8K
QUANT: q2_K
TOOL: UNVERIFIED
AGENT SCORE: 0
COMPATIBLE: False
DESC: Meta Llama 3: The most capable openly available LLM to date
URL: https://ollama.com/library/llama3

════════════════════════════════════════════════════════════════════════════════════════════════════
MODEL: deepseek-r1
ID: 31
════════════════════════════════════════════════════════════════════════════════════════════════════
SIZE: 5.2GB
CONTEXT: 128K
QUANT: q4_K_M
TOOL: PARTIAL
AGENT SCORE: 55
COMPATIBLE: True
DESC: DeepSeek-R1 is a family of open reasoning models with performance approaching that of leading models, such as O3 and Gemini 2.5 Pro.
URL: https://ollama.com/library/deepseek-r1

🛑 Stopping Ngrok...
✅ Ngrok Stopped!


4. Add compactible point row &
Add url into model's model

In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - ULTIMATE PATCHED FULL SRC
#
# FEATURES
# ==============================================================================
# ✅ Ollama Library Crawl
# ✅ HuggingFace GGUF Crawl
# ✅ LM Studio Crawl
# ✅ Provider Sort
# ✅ Pagination
# ✅ Page Jump
# ✅ Delete by ID
# ✅ Delete by Name
# ✅ Info Command
# ✅ Info shows URL
# ✅ Context Detection
# ✅ Quant Detection
# ✅ Compatibility Scoring
# ✅ URL Metadata
# ✅ Realtime Pull Logs
# ✅ Ngrok
# ✅ Stop Server
# ✅ Stop Ngrok
# ✅ Colab Widgets UI
# ✅ Keeps Original UI Logic
# ==============================================================================

import os
import re
import time
import math
import requests
import subprocess

from urllib.parse import quote, unquote

import ipywidgets as widgets

from IPython.display import display, clear_output

# ==============================================================================
# CONFIG
# ==============================================================================

OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_PORT = "11434"

MODELS_PER_PAGE = 35
CURRENT_PAGE = 0

MAX_LIBRARY_PAGES = 25
HTTP_TIMEOUT = 30

HF_MAX_MODELS = 1200

LMSTUDIO_PUBLISHERS = [

    "lmstudio-community",

    "bartowski",

    "QuantFactory",

    "unsloth",

]

# ==============================================================================
# STATE
# ==============================================================================

STATE = {

    "installed": [],

    "library": [],

    "details": {},

}

# ==============================================================================
# MODEL DB
# ==============================================================================

MODEL_DB = {

    "llama3.1": {
        "s": "4.7GB",
        "c": "128K",
        "n": "General assistant, multilingual, strong Vietnamese support."
    },

    "llama3.2": {
        "s": "2.0GB",
        "c": "128K",
        "n": "Lightweight and fast."
    },

    "deepseek-r1": {
        "s": "4.7GB+",
        "c": "128K",
        "n": "Excellent reasoning and coding."
    },

    "qwen2.5-coder": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Coding optimized."
    },

    "phi4": {
        "s": "8GB",
        "c": "128K",
        "n": "Strong logic and reasoning."
    },

    "gemma2": {
        "s": "5.5GB",
        "c": "8K",
        "n": "Google model."
    },

    "mistral": {
        "s": "4.1GB",
        "c": "32K",
        "n": "Stable chatbot."
    },

}

# ==============================================================================
# CMD
# ==============================================================================

def run_cmd(args, timeout=120):

    try:

        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return (
            p.returncode,
            p.stdout.strip(),
            p.stderr.strip()
        )

    except Exception as e:

        return (
            1,
            "",
            str(e)
        )

# ==============================================================================
# CURL
# ==============================================================================

def curl_get(url):

    code, out, err = run_cmd([

        "curl",

        "-L",

        "-sS",

        "--max-time",

        str(HTTP_TIMEOUT),

        url

    ])

    if code != 0:
        return ""

    return out

# ==============================================================================
# SIZE FORMAT
# ==============================================================================

def format_bytes(size):

    try:

        size = int(size)

    except:
        return "---"

    gb = size / (1024 ** 3)

    if gb >= 1:
        return f"{gb:.1f}GB"

    mb = size / (1024 ** 2)

    return f"{mb:.0f}MB"

# ==============================================================================
# PROVIDER SORT
# ==============================================================================

def get_provider_priority(name):

    if not name.startswith("hf.co/") and not name.startswith("lmstudio/"):
        return 0

    if name.startswith("lmstudio/"):
        return 1

    if name.startswith("hf.co/"):
        return 2

    return 999

# ==============================================================================
# COMPAT SCORE
# ==============================================================================

def evaluate_compatibility(model_name, gguf_files):

    score = 0

    reasons = []

    lower_files = [x.lower() for x in gguf_files]

    if gguf_files:
        score += 35
        reasons.append("GGUF")

    quant_keywords = [

        "q4_k_m",
        "q5_k_m",
        "q8_0",
        "q4_0",
        "q6_k",

    ]

    if any(
        q in f
        for q in quant_keywords
        for f in lower_files
    ):
        score += 30
        reasons.append("QUANT")

    if any(
        "instruct" in f
        or "chat" in f
        for f in lower_files
    ):
        score += 15
        reasons.append("CHAT")

    if any(
        "gguf" in f
        for f in lower_files
    ):
        score += 20
        reasons.append("OLLAMA_OK")

    compatible = score >= 60

    return compatible, score, ", ".join(reasons)

# ==============================================================================
# RESTART SERVER
# ==============================================================================

def restart_ollama_server():

    print("🔄 Starting Ollama...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    os.environ["OLLAMA_HOST"] = OLLAMA_HOST

    log_file = open("ollama.log", "w")

    subprocess.Popen(

        ["ollama", "serve"],

        stdout=log_file,

        stderr=log_file

    )

    time.sleep(6)

    print("✅ Ollama Ready!")

# ==============================================================================
# STOP SERVER
# ==============================================================================

def stop_ollama_server():

    print("\n🛑 Stopping Ollama...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    print("✅ Ollama Stopped!")

# ==============================================================================
# NGROK
# ==============================================================================

def run_ngrok():

    print("\n🚀 Starting Ngrok...")

    run_cmd([
        "bash",
        "-lc",
        f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"
    ])

    time.sleep(3)

    try:

        response = requests.get(
            "http://localhost:4040/api/tunnels",
            timeout=5
        )

        tunnels = response.json().get("tunnels", [])

        if tunnels:

            url = tunnels[0]["public_url"]

            print("\n✅ NGROK ONLINE:")
            print(url)

        else:

            print("⚠️ No Tunnel Found")

    except:

        print("⚠️ Ngrok Error")

# ==============================================================================
# STOP NGROK
# ==============================================================================

def stop_ngrok():

    print("\n🛑 Stopping Ngrok...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ngrok || true"
    ])

    print("✅ Ngrok Stopped!")

# ==============================================================================
# INSTALLED MODELS
# ==============================================================================

def get_installed_models():

    code, out, err = run_cmd(
        ["ollama", "list"],
        timeout=20
    )

    if code != 0:
        return []

    lines = out.splitlines()

    if len(lines) <= 1:
        return []

    models = []

    for line in lines[1:]:

        parts = line.split()

        if not parts:
            continue

        name = parts[0].split(":")[0]

        models.append(name)

    return models

# ==============================================================================
# FETCH OLLAMA LIBRARY
# ==============================================================================

def fetch_library_models():

    models = []

    for page in range(1, MAX_LIBRARY_PAGES + 1):

        try:

            if page == 1:
                url = "https://ollama.com/library"
            else:
                url = f"https://ollama.com/library?page={page}"

            html = curl_get(url)

            if not html:
                continue

            found = re.findall(
                r'href="/library/([^"]+)"',
                html
            )

            for item in found:

                item = unquote(item)

                if "/" in item:
                    continue

                if item not in models:
                    models.append(item)

        except:
            pass

    return models

# ==============================================================================
# FETCH OLLAMA DETAILS
# ==============================================================================

def fetch_ollama_details(slug):

    result = {

        "size": "---",
        "context": "---",
        "description": "",
        "url": f"https://ollama.com/library/{slug}",
        "compat": True,
        "score": 100,
        "reason": "Official Ollama"

    }

    try:

        html = curl_get(
            f"https://ollama.com/library/{quote(slug)}"
        )

        if html:

            desc = re.search(
                r'<meta name="description" content="([^"]+)"',
                html
            )

            if desc:
                result["description"] = desc.group(1)

        tags_html = curl_get(
            f"https://ollama.com/library/{quote(slug)}/tags"
        )

        if tags_html:

            text = re.sub(
                r"<[^>]+>",
                " ",
                tags_html
            )

            size_match = re.search(
                r'([0-9.]+\s?(?:GB|MB))',
                text
            )

            ctx_match = re.search(
                r'([0-9.]+\s?[KMG]?)\s*context',
                text,
                flags=re.I
            )

            if size_match:
                result["size"] = size_match.group(1)

            if ctx_match:
                result["context"] = ctx_match.group(1)

    except:
        pass

    return result

# ==============================================================================
# FETCH HF
# ==============================================================================

def fetch_huggingface_models():

    models = []

    try:

        url = (
            "https://huggingface.co/api/models"
            "?search=GGUF"
            # "&limit=999999"
            "&full=true"
        )

        response = requests.get(
            url,
            timeout=40
        )

        data = response.json()

        for item in data:

            model_id = item.get("id", "")

            if not model_id:
                continue

            siblings = item.get("siblings", [])

            gguf_files = []

            total_size = 0

            for s in siblings:

                fname = s.get(
                    "rfilename",
                    ""
                )

                if fname.lower().endswith(".gguf"):

                    gguf_files.append(fname)

                    size = s.get("size")

                    if isinstance(size, int):
                        total_size += size

            if not gguf_files:
                continue

            compat, score, reason = evaluate_compatibility(
                model_id,
                gguf_files
            )

            quant = "---"

            for q in [

                "q2_k",
                "q3_k",
                "q4_0",
                "q4_k_m",
                "q5_k_m",
                "q6_k",
                "q8_0"

            ]:

                if any(q in x.lower() for x in gguf_files):

                    quant = q.upper()
                    break

            desc = item.get("description", "")

            models.append({

                "name": f"hf.co/{model_id}",

                "quant": quant,

                "compatible": compat,

                "score": score,

                "reason": reason,

                "size": format_bytes(total_size),

                "context": "---",

                "description": desc[:120],

                "url": f"https://huggingface.co/{model_id}"

            })

            if len(models) >= HF_MAX_MODELS:
                break

    except Exception as e:

        print(f"HF ERROR: {e}")

    return models

# ==============================================================================
# FETCH LM STUDIO
# ==============================================================================

def fetch_lmstudio_models():

    models = []

    try:

        for publisher in LMSTUDIO_PUBLISHERS:

            url = (
                f"https://huggingface.co/api/models"
                f"?author={publisher}"
                # f"&limit=1000"
                f"&full=true"
            )

            response = requests.get(
                url,
                timeout=40
            )

            data = response.json()

            for item in data:

                model_id = item.get("id", "")

                if not model_id:
                    continue

                siblings = item.get("siblings", [])

                gguf_files = []

                total_size = 0

                for s in siblings:

                    fname = s.get(
                        "rfilename",
                        ""
                    )

                    if fname.lower().endswith(".gguf"):

                        gguf_files.append(fname)

                        size = s.get("size")

                        if isinstance(size, int):
                            total_size += size

                if not gguf_files:
                    continue

                compat, score, reason = evaluate_compatibility(
                    model_id,
                    gguf_files
                )

                quant = "---"

                for q in [

                    "q2_k",
                    "q3_k",
                    "q4_0",
                    "q4_k_m",
                    "q5_k_m",
                    "q6_k",
                    "q8_0"

                ]:

                    if any(q in x.lower() for x in gguf_files):

                        quant = q.upper()
                        break

                desc = item.get("description", "")

                models.append({

                    "name": f"lmstudio/{model_id}",

                    "quant": quant,

                    "compatible": compat,

                    "score": score,

                    "reason": reason,

                    "size": format_bytes(total_size),

                    "context": "---",

                    "description": desc[:120],

                    "url": f"https://huggingface.co/{model_id}"

                })

    except Exception as e:

        print(f"LM ERROR: {e}")

    return models

# ==============================================================================
# REFRESH
# ==============================================================================

def refresh_all_data():

    STATE["installed"] = get_installed_models()

    ollama_models = fetch_library_models()

    hf_models = fetch_huggingface_models()

    lm_models = fetch_lmstudio_models()

    merged = []

    merged.extend(ollama_models)

    merged.extend([
        x["name"]
        for x in hf_models
    ])

    merged.extend([
        x["name"]
        for x in lm_models
    ])

    merged = list(dict.fromkeys(merged))

    merged = sorted(

        merged,

        key=lambda x: (
            get_provider_priority(x),
            x.lower()
        )

    )

    STATE["library"] = merged

    STATE["details"] = {}

    # OLLAMA DETAILS
    for slug in ollama_models:

        STATE["details"][slug] = fetch_ollama_details(slug)

    # HF + LM DETAILS
    for item in hf_models + lm_models:

        STATE["details"][item["name"]] = {

            "compat": item["compatible"],

            "score": item["score"],

            "reason": item["reason"],

            "quant": item["quant"],

            "size": item["size"],

            "context": item["context"],

            "description": item["description"],

            "url": item["url"]

        }

# ==============================================================================
# PULL
# ==============================================================================

def pull_model(model_name):

    print("\n" + "═" * 100)

    print(f"📥 DOWNLOADING: {model_name}")

    print("═" * 100)

    try:

        process = subprocess.Popen(

            ["ollama", "pull", model_name],

            stdout=subprocess.PIPE,

            stderr=subprocess.STDOUT,

            text=True,

            bufsize=1

        )

        last_line = ""

        while True:

            line = process.stdout.readline()

            if not line and process.poll() is not None:
                break

            if not line:
                continue

            line = line.strip()

            if not line:
                continue

            pretty = line

            pretty = pretty.replace(
                "pulling manifest",
                "📦 pulling manifest"
            )

            pretty = pretty.replace(
                "verifying sha256 digest",
                "🔐 verifying digest"
            )

            pretty = pretty.replace(
                "writing manifest",
                "💾 writing manifest"
            )

            pretty = pretty.replace(
                "success",
                "✅ success"
            )

            if "%" in pretty:
                pretty = "⬇️ " + pretty

            if pretty != last_line:

                print(pretty)

                last_line = pretty

        rc = process.poll()

        print("\n" + "═" * 100)

        if rc == 0:
            print("✅ DOWNLOAD SUCCESS")
        else:
            print("⚠️ DOWNLOAD FAILED")

    except Exception as e:

        print(f"❌ ERROR: {e}")

# ==============================================================================
# DELETE
# ==============================================================================

def delete_model(model_name):

    print(f"\n🗑️ Removing: {model_name}")

    run_cmd([
        "ollama",
        "rm",
        model_name
    ])

    print("✅ Removed!")

# ==============================================================================
# INFO
# ==============================================================================

def show_model_info(model_name):

    detail = STATE["details"].get(
        model_name,
        {}
    )

    model_id = None

    try:

        model_id = (
            STATE["library"].index(model_name)
            + 1
        )
    except:
        pass

    print("\n" + "═" * 100)

    print(f"MODEL: {model_name}")

    if model_id:
        print(f"ID: {model_id}")

    print("═" * 100)

    print(f"SIZE: {detail.get('size', '---')}")
    print(f"CONTEXT: {detail.get('context', '---')}")
    print(f"QUANT: {detail.get('quant', '---')}")
    print(f"COMPATIBLE: {detail.get('compat', '---')}")
    print(f"SCORE: {detail.get('score', '---')}/100")
    print(f"REASON: {detail.get('reason', '---')}")
    print(f"DESC: {detail.get('description', '---')}")
    print(f"URL: {detail.get('url', '---')}")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================

def process_command(choice):

    global CURRENT_PAGE

    low = choice.lower()

    if low.startswith("page "):

        try:

            page_num = int(
                low.split()[1]
            )

            max_page = max(
                1,
                math.ceil(
                    len(STATE["library"])
                    / MODELS_PER_PAGE
                )
            )

            if 1 <= page_num <= max_page:

                CURRENT_PAGE = page_num - 1

                render_catalog()

        except:

            print("⚠️ PAGE ERROR")

        return

    if low.startswith('delete "'):

        try:

            model_name = re.findall(

                r'delete\s+"(.+)"',

                choice,

                flags=re.I

            )[0]

            delete_model(model_name)

            refresh_all_data()

            render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    if low.startswith("delete "):

        try:

            delete_id = int(
                choice.split()[1]
            )

            idx = delete_id - 101

            if 0 <= idx < len(STATE["installed"]):

                target = STATE["installed"][idx]

                delete_model(target)

                refresh_all_data()

                render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    if low.startswith("info "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            idx = int(arg) - 1

            if 0 <= idx < len(STATE["library"]):

                show_model_info(
                    STATE["library"][idx]
                )

        else:

            show_model_info(arg)

        return

    if low == "refresh":

        refresh_all_data()

        render_catalog()

        return

    if low == "stop ngrok":

        stop_ngrok()

        return

    if low == "stop server":

        stop_ollama_server()

        return

    if low == "0":

        run_ngrok()

        return

    if choice.isdigit():

        idx = int(choice) - 1

        if 0 <= idx < len(STATE["library"]):

            target = STATE["library"][idx]

            pull_model(target)

            refresh_all_data()

            render_catalog()

        return

    pull_model(choice)

    refresh_all_data()

    render_catalog()

# ==============================================================================
# RENDER
# ==============================================================================

def render_catalog():

    with OUTPUT:

        clear_output(wait=True)

        total_models = len(
            STATE["library"]
        )

        max_page = max(
            1,
            math.ceil(
                total_models
                / MODELS_PER_PAGE
            )
        )

        start = CURRENT_PAGE * MODELS_PER_PAGE

        end = start + MODELS_PER_PAGE

        page_models = STATE["library"][start:end]

        print("═" * 190)

        print(
            "ID   | SRC  | STATUS         | MODEL                              | SIZE     | CTX     | QUANT   | COMPAT | DESCRIPTION"
        )

        print("─" * 190)

        installed = set(
            STATE["installed"]
        )

        for offset, slug in enumerate(
            page_models
        ):

            i = start + offset + 1

            detail = STATE["details"].get(
                slug,
                {}
            )

            compat = detail.get("compat")

            quant = detail.get("quant", "---")

            context = detail.get("context", "---")

            if slug.startswith("hf.co/"):

                if compat is True:
                    source = "🤗🟢"

                elif compat is False:
                    source = "🤗🔴"

                else:
                    source = "🤗"

            elif slug.startswith("lmstudio/"):

                if compat is True:
                    source = "🧠🟢"

                elif compat is False:
                    source = "🧠🔴"

                else:
                    source = "🧠"

            else:

                source = "🦙"

            if slug in installed:
                status = "✅ INSTALLED"
            else:
                status = "⚪ EMPTY"

            size = detail.get("size", "---")

            desc = detail.get(
                "description",
                ""
            )

            if len(desc) > 55:
                desc = desc[:55] + "..."

            compat_text = detail.get(
                "score",
                "---"
            )

            display_slug = slug[:34]

            print(

                f"{str(i):<4} | "

                f"{source:<5} | "

                f"{status:<14} | "

                f"{display_slug:<34} | "

                f"{size:<8} | "

                f"{context:<7} | "

                f"{quant:<8} | "

                f"{str(compat_text):<7} | "

                f"{desc}"

            )

        print("\n📂 INSTALLED MODELS:")

        for i, name in enumerate(

            STATE["installed"],

            101

        ):

            print(
                f"{i:<4} | 🗑️ {name}"
            )

        print("\n" + "═" * 190)

        ollama_count = len([
            x for x in STATE["library"]
            if not x.startswith("hf.co/")
            and not x.startswith("lmstudio/")
        ])

        lmstudio_count = len([
            x for x in STATE["library"]
            if x.startswith("lmstudio/")
        ])

        hf_count = len([
            x for x in STATE["library"]
            if x.startswith("hf.co/")
        ])

        print(
            f"🦙 Ollama: {ollama_count} | "
            f"🧠 LM Studio: {lmstudio_count} | "
            f"🤗 HF: {hf_count}"
        )

        print(
            f"\n📄 PAGE: "
            f"{CURRENT_PAGE + 1}/{max_page}"
        )

        print("\nCOMMANDS:")

        print("1                    -> pull model")
        print("delete 101           -> delete by id")
        print('delete "llama3"      -> delete by name')
        print("info 1               -> model info")
        print("page 5               -> jump page")
        print("refresh              -> refresh")
        print("stop ngrok           -> stop ngrok")
        print("stop server          -> stop ollama")
        print("0                    -> start ngrok")

# ==============================================================================
# SUBMIT
# ==============================================================================

def on_submit(_=None):

    cmd = INPUT_BOX.value.strip()

    INPUT_BOX.value = ""

    if not cmd:
        return

    with OUTPUT:

        print(f"\n👉 {cmd}")

    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================

restart_ollama_server()

refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================

TITLE = widgets.HTML("""
<h2>🚀 Ollama Colab Manager</h2>
""")

HELP = widgets.HTML("""
<b>
Commands:
1 | delete 101 | info 1 | stop ngrok | stop server
</b>
""")

INPUT_BOX = widgets.Text(

    placeholder="Nhập lệnh...",

    description=">>>",

    layout=widgets.Layout(width="90%")

)

SEND_BTN = widgets.Button(

    description="Gửi",

    button_style="success"

)

REFRESH_BTN = widgets.Button(

    description="Refresh",

    button_style="info"

)

NGROK_BTN = widgets.Button(

    description="Start Ngrok",

    button_style="warning"

)

STOP_NGROK_BTN = widgets.Button(

    description="Stop Ngrok",

    button_style="danger"

)

STOP_SERVER_BTN = widgets.Button(

    description="Stop Server",

    button_style="danger"

)

OUTPUT = widgets.Output()

# ==============================================================================
# EVENTS
# ==============================================================================

SEND_BTN.on_click(on_submit)

INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(

    lambda _: (
        refresh_all_data(),
        render_catalog()
    )

)

NGROK_BTN.on_click(
    lambda _: run_ngrok()
)

STOP_NGROK_BTN.on_click(
    lambda _: stop_ngrok()
)

STOP_SERVER_BTN.on_click(
    lambda _: stop_ollama_server()
)

# ==============================================================================
# BUTTONS
# ==============================================================================

BUTTONS = widgets.HBox([

    SEND_BTN,

    REFRESH_BTN,

    NGROK_BTN,

    STOP_NGROK_BTN,

    STOP_SERVER_BTN

])

# ==============================================================================
# MAIN UI
# ==============================================================================

UI = widgets.VBox([

    TITLE,

    OUTPUT,

    HELP,

    INPUT_BOX,

    BUTTONS

])

display(UI)

render_catalog()

🔄 Starting Ollama...
✅ Ollama Ready!



════════════════════════════════════════════════════════════════════════════════════════════════════
MODEL: all-minilm
ID: 2
════════════════════════════════════════════════════════════════════════════════════════════════════
SIZE: 46MB
CONTEXT: 512 
QUANT: ---
COMPATIBLE: True
SCORE: 100/100
REASON: Official Ollama
DESC: Embedding models on very large sentence level datasets.
URL: https://ollama.com/library/all-minilm

════════════════════════════════════════════════════════════════════════════════════════════════════
MODEL: hf.co/TrevorJS/voxtral-mini-realtime-gguf
ID: 3213
════════════════════════════════════════════════════════════════════════════════════════════════════
SIZE: 0MB
CONTEXT: ---
QUANT: ---
COMPATIBLE: False
SCORE: 55/100
REASON: GGUF, OLLAMA_OK
DESC: 
URL: https://huggingface.co/TrevorJS/voxtral-mini-realtime-gguf


5. Ollama lib + hugging face + LMStudio model

In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - FULL SRC
# PATCHED + ENHANCED VERSION
#
# FEATURES:
# ✅ Ollama Library
# ✅ HuggingFace GGUF Crawl
# ✅ LM Studio Crawl
# ✅ Page System
# ✅ Delete by ID
# ✅ Delete by Name
# ✅ Info with URL
# ✅ Context Window
# ✅ Quant Detection
# ✅ Compatibility Check
# ✅ Description
# ✅ Realtime Pull Logs
# ✅ Ngrok
# ==============================================================================

import os
import re
import time
import json
import requests
import subprocess

from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# CONFIG
# ==============================================================================

OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_PORT = "11434"

MODELS_PER_PAGE = 35
CURRENT_PAGE = 0

MAX_LIBRARY_PAGES = 25
HTTP_TIMEOUT = 30

HF_MAX_MODELS = 1000

LMSTUDIO_PUBLISHERS = [

    "lmstudio-community",

    "bartowski",

    "QuantFactory",

    "unsloth",

]

# ==============================================================================
# STATE
# ==============================================================================

STATE = {

    "installed": [],

    "library": [],

    "details": {},

}

# ==============================================================================
# MODEL DB
# ==============================================================================

MODEL_DB = {

    "llama3.1": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Meta flagship multilingual model."
    },

    "llama3.2": {
        "s": "2.0GB",
        "c": "128K",
        "n": "Lightweight model for weaker devices."
    },

    "deepseek-r1": {
        "s": "4.7GB+",
        "c": "128K",
        "n": "Excellent reasoning and coding."
    },

    "qwen2.5-coder": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Optimized for developers."
    },

    "phi4": {
        "s": "8GB",
        "c": "128K",
        "n": "Strong logic and math."
    },

    "gemma2": {
        "s": "5GB",
        "c": "8K",
        "n": "Google lightweight model."
    },

}

# ==============================================================================
# CMD
# ==============================================================================

def run_cmd(args, timeout=120):

    try:

        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return (
            p.returncode,
            p.stdout.strip(),
            p.stderr.strip()
        )

    except Exception as e:

        return (
            1,
            "",
            str(e)
        )

# ==============================================================================
# CURL
# ==============================================================================

def curl_get(url):

    code, out, err = run_cmd([

        "curl",

        "-L",

        "-sS",

        "--max-time",

        str(HTTP_TIMEOUT),

        url

    ])

    if code != 0:
        return ""

    return out

# ==============================================================================
# PROVIDER PRIORITY
# ==============================================================================

def get_provider_priority(name):

    if not name.startswith("hf.co/") and not name.startswith("lmstudio/"):
        return 0

    if name.startswith("lmstudio/"):
        return 1

    if name.startswith("hf.co/"):
        return 2

    return 999

# ==============================================================================
# SERVER
# ==============================================================================

def restart_ollama_server():

    print("🔄 Starting Ollama...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    os.environ["OLLAMA_HOST"] = OLLAMA_HOST

    log_file = open("ollama.log", "w")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=log_file
    )

    time.sleep(6)

    print("✅ Ollama Ready!")

# ==============================================================================
# STOP SERVER
# ==============================================================================

def stop_ollama_server():

    print("\n🛑 Stopping Ollama...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    print("✅ Ollama Stopped!")

# ==============================================================================
# NGROK
# ==============================================================================

def run_ngrok():

    print("\n🚀 Starting Ngrok...")

    run_cmd([
        "bash",
        "-lc",
        f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"
    ])

    time.sleep(3)

    try:

        response = requests.get(
            "http://localhost:4040/api/tunnels",
            timeout=5
        )

        tunnels = response.json().get("tunnels", [])

        if tunnels:

            url = tunnels[0]["public_url"]

            print("\n✅ NGROK ONLINE:")
            print(url)

        else:

            print("⚠️ No Tunnel Found")

    except:

        print("⚠️ Ngrok Error")

# ==============================================================================
# STOP NGROK
# ==============================================================================

def stop_ngrok():

    print("\n🛑 Stopping Ngrok...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ngrok || true"
    ])

    print("✅ Ngrok Stopped!")

# ==============================================================================
# INSTALLED MODELS
# ==============================================================================

def get_installed_models():

    code, out, err = run_cmd(
        ["ollama", "list"],
        timeout=20
    )

    if code != 0:
        return []

    lines = out.splitlines()

    if len(lines) <= 1:
        return []

    models = []

    for line in lines[1:]:

        parts = line.split()

        if not parts:
            continue

        name = parts[0].split(":")[0]

        models.append(name)

    return models

# ==============================================================================
# OLLAMA LIBRARY
# ==============================================================================

def fetch_library_models():

    models = []

    for page in range(1, MAX_LIBRARY_PAGES + 1):

        try:

            if page == 1:
                url = "https://ollama.com/library"
            else:
                url = f"https://ollama.com/library?page={page}"

            html = curl_get(url)

            if not html:
                continue

            found = re.findall(
                r'href="/library/([^"]+)"',
                html
            )

            for item in found:

                item = unquote(item)

                if "/" in item:
                    continue

                if item not in models:
                    models.append(item)

        except:
            pass

    return models

# ==============================================================================
# OLLAMA DETAIL
# ==============================================================================

def fetch_ollama_detail(slug):

    result = {

        "size": "---",

        "context": "---",

        "description": "",

        "compat": True,

        "quant": "---",

        "url": f"https://ollama.com/library/{slug}"

    }

    try:

        html = curl_get(
            f"https://ollama.com/library/{quote(slug)}"
        )

        if html:

            desc = re.search(
                r'<meta name="description" content="([^"]+)"',
                html
            )

            if desc:
                result["description"] = desc.group(1)

        tags_html = curl_get(
            f"https://ollama.com/library/{quote(slug)}/tags"
        )

        if tags_html:

            text = re.sub(
                r"<[^>]+>",
                " ",
                tags_html
            )

            size_match = re.search(
                r'([0-9.]+\s?(?:GB|MB))',
                text
            )

            ctx_match = re.search(
                r'([0-9.]+\s?[KMG]?)\s*context',
                text,
                flags=re.I
            )

            if size_match:
                result["size"] = size_match.group(1)

            if ctx_match:
                result["context"] = ctx_match.group(1)

    except:
        pass

    return result

# ==============================================================================
# HF MODELS
# ==============================================================================

def fetch_huggingface_models():

    models = []

    try:

        url = (
            "https://huggingface.co/api/models"
            "?search=GGUF"
            # "&limit=1000"
            "&full=true"
        )

        response = requests.get(
            url,
            timeout=40
        )

        data = response.json()

        for item in data:

            model_id = item.get("id", "")

            if not model_id:
                continue

            siblings = item.get("siblings", [])

            gguf_files = []

            total_size = 0

            for s in siblings:

                fname = s.get(
                    "rfilename",
                    ""
                ).lower()

                if fname.endswith(".gguf"):

                    gguf_files.append(fname)

                    size = s.get("size")

                    if isinstance(size, int):
                        total_size += size

            if not gguf_files:
                continue

            quant = "---"

            for q in [

                "q2_k",
                "q3_k",
                "q4_0",
                "q4_k_m",
                "q5_k_m",
                "q6_k",
                "q8_0"

            ]:

                if any(q in x for x in gguf_files):

                    quant = q.upper()

                    break

            size_text = "---"

            if total_size > 0:

                gb = total_size / (1024 ** 3)

                if gb >= 1:
                    size_text = f"{gb:.1f}GB"
                else:
                    mb = total_size / (1024 ** 2)
                    size_text = f"{mb:.0f}MB"

            context = "---"

            for f in gguf_files:

                ctx = re.search(
                    r'(\d+)[Kk]',
                    f
                )

                if ctx:
                    context = ctx.group(1) + "K"
                    break

            models.append({

                "name": f"hf.co/{model_id}",

                "quant": quant,

                "compatible": True,

                "reason": item.get(
                    "pipeline_tag",
                    "HF GGUF"
                ),

                "size": size_text,

                "context": context,

                "url": f"https://huggingface.co/{model_id}"

            })

            if len(models) >= HF_MAX_MODELS:
                break

    except Exception as e:

        print(f"HF ERROR: {e}")

    return models

# ==============================================================================
# LM STUDIO
# ==============================================================================

def fetch_lmstudio_models():

    models = []

    try:

        for publisher in LMSTUDIO_PUBLISHERS:

            url = (
                f"https://huggingface.co/api/models"
                f"?author={publisher}"
                # f"&limit=1000"
                f"&full=true"
            )

            response = requests.get(
                url,
                timeout=40
            )

            data = response.json()

            for item in data:

                model_id = item.get("id", "")

                if not model_id:
                    continue

                siblings = item.get("siblings", [])

                gguf_files = []

                total_size = 0

                for s in siblings:

                    fname = s.get(
                        "rfilename",
                        ""
                    ).lower()

                    if fname.endswith(".gguf"):

                        gguf_files.append(fname)

                        size = s.get("size")

                        if isinstance(size, int):
                            total_size += size

                if not gguf_files:
                    continue

                quant = "---"

                for q in [

                    "q2_k",
                    "q3_k",
                    "q4_0",
                    "q4_k_m",
                    "q5_k_m",
                    "q6_k",
                    "q8_0"

                ]:

                    if any(q in x for x in gguf_files):

                        quant = q.upper()

                        break

                size_text = "---"

                if total_size > 0:

                    gb = total_size / (1024 ** 3)

                    if gb >= 1:
                        size_text = f"{gb:.1f}GB"
                    else:
                        mb = total_size / (1024 ** 2)
                        size_text = f"{mb:.0f}MB"

                context = "---"

                for f in gguf_files:

                    ctx = re.search(
                        r'(\d+)[Kk]',
                        f
                    )

                    if ctx:
                        context = ctx.group(1) + "K"
                        break

                models.append({

                    "name": f"lmstudio/{model_id}",

                    "quant": quant,

                    "compatible": True,

                    "reason": item.get(
                        "pipeline_tag",
                        "LM Studio"
                    ),

                    "size": size_text,

                    "context": context,

                    "url": f"https://huggingface.co/{model_id}"

                })

    except Exception as e:

        print(f"LM ERROR: {e}")

    return models

# ==============================================================================
# REFRESH
# ==============================================================================

def refresh_all_data():

    STATE["installed"] = get_installed_models()

    ollama_models = fetch_library_models()

    hf_models = fetch_huggingface_models()

    lm_models = fetch_lmstudio_models()

    merged = []

    merged.extend(ollama_models)

    merged.extend([
        x["name"]
        for x in hf_models
    ])

    merged.extend([
        x["name"]
        for x in lm_models
    ])

    merged = list(dict.fromkeys(merged))

    merged = sorted(
        merged,
        key=lambda x: (
            get_provider_priority(x),
            x.lower()
        )
    )

    STATE["library"] = merged

    STATE["details"] = {}

    # OLLAMA DETAIL
    for model in ollama_models:

        STATE["details"][model] = fetch_ollama_detail(model)

    # HF + LM
    for item in hf_models + lm_models:

        STATE["details"][item["name"]] = {

            "compat": item["compatible"],

            "reason": item["reason"],

            "quant": item["quant"],

            "size": item["size"],

            "context": item["context"],

            "description": item["reason"],

            "url": item["url"]

        }

# ==============================================================================
# PULL
# ==============================================================================

def pull_model(model_name):

    print("\n" + "═" * 100)

    print(f"📥 DOWNLOADING: {model_name}")

    print("═" * 100)

    try:

        process = subprocess.Popen(
            ["ollama", "pull", model_name],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        last_line = ""

        while True:

            line = process.stdout.readline()

            if not line and process.poll() is not None:
                break

            if not line:
                continue

            line = line.strip()

            if not line:
                continue

            pretty = line

            pretty = pretty.replace(
                "pulling manifest",
                "📦 pulling manifest"
            )

            pretty = pretty.replace(
                "verifying sha256 digest",
                "🔐 verifying digest"
            )

            pretty = pretty.replace(
                "writing manifest",
                "💾 writing manifest"
            )

            pretty = pretty.replace(
                "success",
                "✅ success"
            )

            if "%" in pretty:
                pretty = "⬇️ " + pretty

            if pretty != last_line:

                print(pretty)

                last_line = pretty

        rc = process.poll()

        print("\n" + "═" * 100)

        if rc == 0:
            print("✅ DOWNLOAD SUCCESS")
        else:
            print("⚠️ DOWNLOAD FAILED")

    except Exception as e:

        print(f"❌ ERROR: {e}")

# ==============================================================================
# DELETE
# ==============================================================================

def delete_model(model_name):

    print(f"\n🗑️ Removing: {model_name}")

    run_cmd([
        "ollama",
        "rm",
        model_name
    ])

    print("✅ Removed!")

# ==============================================================================
# INFO
# ==============================================================================

def show_model_info(model_name):

    detail = STATE["details"].get(
        model_name,
        {}
    )

    model_id = None

    try:
        model_id = (
            STATE["library"].index(model_name)
            + 1
        )
    except:
        pass

    print("\n" + "═" * 100)

    print(f"MODEL: {model_name}")

    if model_id:
        print(f"ID: {model_id}")

    print("═" * 100)

    print(f"SIZE: {detail.get('size', '---')}")
    print(f"CONTEXT: {detail.get('context', '---')}")
    print(f"QUANT: {detail.get('quant', '---')}")
    print(f"COMPATIBLE: {detail.get('compat', '---')}")
    print(f"DESC: {detail.get('description', '---')}")
    print(f"URL: {detail.get('url', '---')}")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================

def process_command(choice):

    global CURRENT_PAGE

    low = choice.lower()

    # PAGE
    if low.startswith("page "):

        try:

            page_num = int(
                low.split()[1]
            )

            max_page = max(
                1,
                (len(STATE["library"]) - 1)
                // MODELS_PER_PAGE + 1
            )

            if 1 <= page_num <= max_page:

                CURRENT_PAGE = page_num - 1

                render_catalog()

        except:

            print("⚠️ PAGE ERROR")

        return

    # DELETE NAME
    if low.startswith('delete "'):

        try:

            model_name = re.findall(
                r'delete\s+"(.+)"',
                choice,
                flags=re.I
            )[0]

            delete_model(model_name)

            refresh_all_data()

            render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    # DELETE ID
    if low.startswith("delete "):

        try:

            delete_id = int(
                choice.split()[1]
            )

            idx = delete_id - 101

            if 0 <= idx < len(STATE["installed"]):

                target = STATE["installed"][idx]

                delete_model(target)

                refresh_all_data()

                render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    # INFO
    if low.startswith("info "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            idx = int(arg) - 1

            if 0 <= idx < len(STATE["library"]):

                show_model_info(
                    STATE["library"][idx]
                )

        else:

            show_model_info(arg)

        return

    # REFRESH
    if low == "refresh":

        refresh_all_data()

        render_catalog()

        return

    # STOP NGROK
    if low == "stop ngrok":

        stop_ngrok()

        return

    # STOP SERVER
    if low == "stop server":

        stop_ollama_server()

        return

    # NGROK
    if low == "0":

        run_ngrok()

        return

    # PULL BY ID
    if choice.isdigit():

        idx = int(choice) - 1

        if 0 <= idx < len(STATE["library"]):

            target = STATE["library"][idx]

            pull_model(target)

            refresh_all_data()

            render_catalog()

        return

    # MANUAL PULL
    pull_model(choice)

    refresh_all_data()

    render_catalog()

# ==============================================================================
# RENDER
# ==============================================================================

def render_catalog():

    with OUTPUT:

        clear_output(wait=True)

        total_models = len(
            STATE["library"]
        )

        max_page = max(
            1,
            (total_models - 1)
            // MODELS_PER_PAGE + 1
        )

        start = CURRENT_PAGE * MODELS_PER_PAGE

        end = start + MODELS_PER_PAGE

        page_models = STATE["library"][start:end]

        print("═" * 180)

        print(
            "ID   | SRC | STATUS         | MODEL                              | SIZE     | CTX      | QUANT    | COMPAT | DESCRIPTION"
        )

        print("─" * 180)

        installed = set(
            STATE["installed"]
        )

        for offset, slug in enumerate(
            page_models
        ):

            i = start + offset + 1

            detail = STATE["details"].get(
                slug,
                {}
            )

            compat = detail.get("compat")
            quant = detail.get("quant", "---")

            if slug.startswith("hf.co/"):

                if compat is True:
                    source = "🤗🟢"

                elif compat is False:
                    source = "🤗🔴"

                else:
                    source = "🤗"

            elif slug.startswith("lmstudio/"):

                if compat is True:
                    source = "🧠🟢"
                else:
                    source = "🧠"

            else:

                source = "🦙"

            if slug in installed:
                status = "✅ INSTALLED"
            else:
                status = "⚪ EMPTY"

            fallback = MODEL_DB.get(
                slug.lower(),
                {}
            )

            size = detail.get(
                "size",
                fallback.get("s", "---")
            )

            context = detail.get(
                "context",
                fallback.get("c", "---")
            )

            desc = detail.get(
                "description",
                fallback.get("n", "")
            )

            if len(desc) > 55:
                desc = desc[:55] + "..."

            compat_text = "YES" if compat else "---"

            display_slug = slug[:34]

            print(

                f"{str(i):<4} | "

                f"{source:<4} | "

                f"{status:<14} | "

                f"{display_slug:<34} | "

                f"{size:<8} | "

                f"{context:<8} | "

                f"{quant:<8} | "

                f"{compat_text:<7} | "

                f"{desc}"

            )

        print("\n📂 INSTALLED MODELS:")

        for i, name in enumerate(

            STATE["installed"],

            101

        ):

            print(
                f"{i:<4} | 🗑️ {name}"
            )

        print("\n" + "═" * 180)

        ollama_count = len([
            x for x in STATE["library"]
            if not x.startswith("hf.co/")
            and not x.startswith("lmstudio/")
        ])

        lmstudio_count = len([
            x for x in STATE["library"]
            if x.startswith("lmstudio/")
        ])

        hf_count = len([
            x for x in STATE["library"]
            if x.startswith("hf.co/")
        ])

        print(
            f"🦙 Ollama: {ollama_count} | "
            f"🧠 LM Studio: {lmstudio_count} | "
            f"🤗 HF: {hf_count}"
        )

        print(
            f"\n📄 PAGE: "
            f"{CURRENT_PAGE + 1}/{max_page}"
        )

        print("\nCOMMANDS:")

        print("1                    -> pull model")
        print("delete 101           -> delete by id")
        print('delete "llama3"      -> delete by name')
        print("info 1               -> model info")
        print("page 5               -> jump page")
        print("refresh              -> refresh")
        print("stop ngrok           -> stop ngrok")
        print("stop server          -> stop ollama")
        print("0                    -> start ngrok")

# ==============================================================================
# SUBMIT
# ==============================================================================

def on_submit(_=None):

    cmd = INPUT_BOX.value.strip()

    INPUT_BOX.value = ""

    if not cmd:
        return

    with OUTPUT:

        print(f"\n👉 {cmd}")

    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================

restart_ollama_server()

refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================

TITLE = widgets.HTML("""
<h2>🚀 Ollama Colab Manager</h2>
""")

HELP = widgets.HTML("""
<b>
Commands:
1 | delete 101 | info 1 | stop ngrok | stop server
</b>
""")

INPUT_BOX = widgets.Text(

    placeholder="Nhập lệnh...",

    description=">>>",

    layout=widgets.Layout(width="90%")

)

SEND_BTN = widgets.Button(

    description="Gửi",

    button_style="success"

)

REFRESH_BTN = widgets.Button(

    description="Refresh",

    button_style="info"

)

NGROK_BTN = widgets.Button(

    description="Start Ngrok",

    button_style="warning"

)

STOP_NGROK_BTN = widgets.Button(

    description="Stop Ngrok",

    button_style="danger"

)

STOP_SERVER_BTN = widgets.Button(

    description="Stop Server",

    button_style="danger"

)

OUTPUT = widgets.Output()

# ==============================================================================
# EVENTS
# ==============================================================================

SEND_BTN.on_click(on_submit)

INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(

    lambda _: (
        refresh_all_data(),
        render_catalog()
    )

)

NGROK_BTN.on_click(
    lambda _: run_ngrok()
)

STOP_NGROK_BTN.on_click(
    lambda _: stop_ngrok()
)

STOP_SERVER_BTN.on_click(
    lambda _: stop_ollama_server()
)

# ==============================================================================
# BUTTONS
# ==============================================================================

BUTTONS = widgets.HBox([

    SEND_BTN,

    REFRESH_BTN,

    NGROK_BTN,

    STOP_NGROK_BTN,

    STOP_SERVER_BTN

])

# ==============================================================================
# MAIN UI
# ==============================================================================

UI = widgets.VBox([

    TITLE,

    OUTPUT,

    HELP,

    INPUT_BOX,

    BUTTONS

])

display(UI)

render_catalog()

6. Ollama + Hugging face + lm studio

In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - FULL SRC
# PATCHED VERSION
#
# FEATURES:
# ✅ Ollama Library
# ✅ HuggingFace GGUF Crawl
# ✅ LM Studio Crawl
# ✅ Page System
# ✅ Delete by ID
# ✅ Delete by Name
# ✅ Info with ID
# ✅ Size Detection
# ✅ Quant Detection
# ✅ Provider Sort
# ✅ Ngrok
# ✅ Realtime Pull Logs
# ==============================================================================

import os
import re
import time
import requests
import subprocess

from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

import ipywidgets as widgets

from IPython.display import display, clear_output

# ==============================================================================
# CONFIG
# ==============================================================================

OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_PORT = "11434"

MODELS_PER_PAGE = 35
CURRENT_PAGE = 0

MAX_LIBRARY_PAGES = 25

HTTP_TIMEOUT = 30

HF_MAX_MODELS = 1000

LMSTUDIO_PUBLISHERS = [

    "lmstudio-community",

    "bartowski",

    "QuantFactory",

    "unsloth",

]

# ==============================================================================
# STATE
# ==============================================================================

STATE = {

    "installed": [],

    "library": [],

    "details": {},

}

# ==============================================================================
# MODEL DB
# ==============================================================================

MODEL_DB = {

    "llama3.1": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Đa năng."
    },

    "llama3.2": {
        "s": "2.0GB",
        "c": "128K",
        "n": "Nhẹ."
    },

    "deepseek-r1": {
        "s": "4.7GB+",
        "c": "128K",
        "n": "Code mạnh."
    },

    "qwen2.5-coder": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Coder."
    },

}

# ==============================================================================
# CMD
# ==============================================================================

def run_cmd(args, timeout=120):

    try:

        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return (
            p.returncode,
            p.stdout.strip(),
            p.stderr.strip()
        )

    except Exception as e:

        return (
            1,
            "",
            str(e)
        )

# ==============================================================================
# CURL
# ==============================================================================

def curl_get(url):

    code, out, err = run_cmd([

        "curl",

        "-L",

        "-sS",

        "--max-time",

        str(HTTP_TIMEOUT),

        url

    ])

    if code != 0:
        return ""

    return out

# ==============================================================================
# PROVIDER SORT
# ==============================================================================

def get_provider_priority(name):

    if not name.startswith("hf.co/") and not name.startswith("lmstudio/"):
        return 0

    if name.startswith("lmstudio/"):
        return 1

    if name.startswith("hf.co/"):
        return 2

    return 999

# ==============================================================================
# RESTART SERVER
# ==============================================================================

def restart_ollama_server():

    print("🔄 Starting Ollama...")

    run_cmd([

        "bash",

        "-lc",

        "pkill ollama || true"

    ])

    os.environ["OLLAMA_HOST"] = OLLAMA_HOST

    log_file = open("ollama.log", "w")

    subprocess.Popen(

        ["ollama", "serve"],

        stdout=log_file,

        stderr=log_file

    )

    time.sleep(6)

    print("✅ Ollama Ready!")

# ==============================================================================
# STOP SERVER
# ==============================================================================

def stop_ollama_server():

    print("\n🛑 Stopping Ollama...")

    run_cmd([

        "bash",

        "-lc",

        "pkill ollama || true"

    ])

    print("✅ Ollama Stopped!")

# ==============================================================================
# NGROK
# ==============================================================================

def run_ngrok():

    print("\n🚀 Starting Ngrok...")

    run_cmd([

        "bash",

        "-lc",

        f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"

    ])

    time.sleep(3)

    try:

        response = requests.get(
            "http://localhost:4040/api/tunnels",
            timeout=5
        )

        tunnels = response.json().get("tunnels", [])

        if tunnels:

            url = tunnels[0]["public_url"]

            print("\n✅ NGROK ONLINE:")
            print(url)

        else:

            print("⚠️ No Tunnel Found")

    except:

        print("⚠️ Ngrok Error")

# ==============================================================================
# STOP NGROK
# ==============================================================================

def stop_ngrok():

    print("\n🛑 Stopping Ngrok...")

    run_cmd([

        "bash",

        "-lc",

        "pkill ngrok || true"

    ])

    print("✅ Ngrok Stopped!")

# ==============================================================================
# INSTALLED MODELS
# ==============================================================================

def get_installed_models():

    code, out, err = run_cmd(

        ["ollama", "list"],

        timeout=20

    )

    if code != 0:
        return []

    lines = out.splitlines()

    if len(lines) <= 1:
        return []

    models = []

    for line in lines[1:]:

        parts = line.split()

        if not parts:
            continue

        name = parts[0].split(":")[0]

        models.append(name)

    return models

# ==============================================================================
# FETCH OLLAMA LIBRARY
# ==============================================================================

def fetch_library_models():

    models = []

    for page in range(1, MAX_LIBRARY_PAGES + 1):

        try:

            if page == 1:
                url = "https://ollama.com/library"
            else:
                url = f"https://ollama.com/library?page={page}"

            html = curl_get(url)

            if not html:
                continue

            found = re.findall(

                r'href="/library/([^"]+)"',

                html

            )

            for item in found:

                item = unquote(item)

                if "/" in item:
                    continue

                if item not in models:
                    models.append(item)

        except:
            pass

    return models

# ==============================================================================
# FETCH HF MODELS
# ==============================================================================

def fetch_huggingface_models():

    models = []

    try:

        url = (
            "https://huggingface.co/api/models"
            "?search=GGUF"
            # "&limit=1000"
            "&full=true"
        )

        response = requests.get(
            url,
            timeout=40
        )

        data = response.json()

        for item in data:

            model_id = item.get("id", "")

            if not model_id:
                continue

            siblings = item.get("siblings", [])

            gguf_files = []

            total_size = 0

            for s in siblings:

                fname = s.get(
                    "rfilename",
                    ""
                ).lower()

                if fname.endswith(".gguf"):

                    gguf_files.append(fname)

                    size = s.get("size")

                    if isinstance(size, int):
                        total_size += size

            if not gguf_files:
                continue

            quant = "---"

            for q in [

                "q2_k",

                "q3_k",

                "q4_0",

                "q4_k_m",

                "q5_k_m",

                "q6_k",

                "q8_0"

            ]:

                if any(q in x for x in gguf_files):

                    quant = q.upper()

                    break

            size_text = "---"

            if total_size > 0:

                gb = total_size / (1024 ** 3)

                if gb >= 1:
                    size_text = f"{gb:.1f}GB"
                else:
                    mb = total_size / (1024 ** 2)
                    size_text = f"{mb:.0f}MB"

            models.append({

                "name": f"hf.co/{model_id}",

                "quant": quant,

                "compatible": True,

                "reason": "HF GGUF",

                "size": size_text

            })

            if len(models) >= HF_MAX_MODELS:
                break

    except Exception as e:

        print(f"HF ERROR: {e}")

    return models

# ==============================================================================
# FETCH LM STUDIO
# ==============================================================================

def fetch_lmstudio_models():

    models = []

    try:

        for publisher in LMSTUDIO_PUBLISHERS:

            url = (
                f"https://huggingface.co/api/models"
                f"?author={publisher}"
                # f"&limit=1000"
                f"&full=true"
            )

            response = requests.get(
                url,
                timeout=40
            )

            data = response.json()

            for item in data:

                model_id = item.get("id", "")

                if not model_id:
                    continue

                siblings = item.get(
                    "siblings",
                    []
                )

                gguf_files = []

                total_size = 0

                for s in siblings:

                    fname = s.get(
                        "rfilename",
                        ""
                    ).lower()

                    if fname.endswith(".gguf"):

                        gguf_files.append(fname)

                        size = s.get("size")

                        if isinstance(size, int):
                            total_size += size

                if not gguf_files:
                    continue

                quant = "---"

                for q in [

                    "q2_k",

                    "q3_k",

                    "q4_0",

                    "q4_k_m",

                    "q5_k_m",

                    "q6_k",

                    "q8_0"

                ]:

                    if any(q in x for x in gguf_files):

                        quant = q.upper()

                        break

                size_text = "---"

                if total_size > 0:

                    gb = total_size / (1024 ** 3)

                    if gb >= 1:
                        size_text = f"{gb:.1f}GB"
                    else:
                        mb = total_size / (1024 ** 2)
                        size_text = f"{mb:.0f}MB"

                models.append({

                    "name": f"lmstudio/{model_id}",

                    "quant": quant,

                    "compatible": True,

                    "reason": "LM Studio",

                    "size": size_text

                })

    except Exception as e:

        print(f"LM ERROR: {e}")

    return models

# ==============================================================================
# REFRESH
# ==============================================================================

def refresh_all_data():

    STATE["installed"] = get_installed_models()

    ollama_models = fetch_library_models()

    hf_models = fetch_huggingface_models()

    lm_models = fetch_lmstudio_models()

    merged = []

    merged.extend(ollama_models)

    merged.extend([
        x["name"]
        for x in hf_models
    ])

    merged.extend([
        x["name"]
        for x in lm_models
    ])

    merged = list(dict.fromkeys(merged))

    merged = sorted(

        merged,

        key=lambda x: (
            get_provider_priority(x),
            x.lower()
        )

    )

    STATE["library"] = merged

    STATE["details"] = {}

    for item in hf_models + lm_models:

        model_name = item["name"]

        STATE["details"][model_name] = {

            "compat": item["compatible"],

            "reason": item["reason"],

            "quant": item["quant"],

            "size": item.get("size", "---")

        }

# ==============================================================================
# PULL
# ==============================================================================

def pull_model(model_name):

    print("\n" + "═" * 100)

    print(f"📥 DOWNLOADING: {model_name}")

    print("═" * 100)

    try:

        process = subprocess.Popen(

            ["ollama", "pull", model_name],

            stdout=subprocess.PIPE,

            stderr=subprocess.STDOUT,

            text=True,

            bufsize=1

        )

        last_line = ""

        while True:

            line = process.stdout.readline()

            if not line and process.poll() is not None:
                break

            if not line:
                continue

            line = line.strip()

            if not line:
                continue

            pretty = line

            pretty = pretty.replace(
                "pulling manifest",
                "📦 pulling manifest"
            )

            pretty = pretty.replace(
                "verifying sha256 digest",
                "🔐 verifying digest"
            )

            pretty = pretty.replace(
                "writing manifest",
                "💾 writing manifest"
            )

            pretty = pretty.replace(
                "success",
                "✅ success"
            )

            if "%" in pretty:
                pretty = "⬇️ " + pretty

            if pretty != last_line:

                print(pretty)

                last_line = pretty

        rc = process.poll()

        print("\n" + "═" * 100)

        if rc == 0:
            print("✅ DOWNLOAD SUCCESS")
        else:
            print("⚠️ DOWNLOAD FAILED")

    except Exception as e:

        print(f"❌ ERROR: {e}")

# ==============================================================================
# DELETE
# ==============================================================================

def delete_model(model_name):

    print(f"\n🗑️ Removing: {model_name}")

    run_cmd([
        "ollama",
        "rm",
        model_name
    ])

    print("✅ Removed!")

# ==============================================================================
# INFO
# ==============================================================================

def show_model_info(model_name):

    detail = STATE["details"].get(
        model_name,
        {}
    )

    model_id = None

    try:
        model_id = (
            STATE["library"].index(model_name)
            + 1
        )
    except:
        pass

    print("\n" + "═" * 100)

    print(f"MODEL: {model_name}")

    if model_id:
        print(f"ID: {model_id}")

    print("═" * 100)

    print(f"SIZE: {detail.get('size', '---')}")
    print(f"QUANT: {detail.get('quant', '---')}")
    print(f"DESC: {detail.get('reason', '---')}")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================

def process_command(choice):

    global CURRENT_PAGE

    low = choice.lower()

    # PAGE
    if low.startswith("page "):

        try:

            page_num = int(
                low.split()[1]
            )

            max_page = max(
                1,
                (len(STATE["library"]) - 1)
                // MODELS_PER_PAGE + 1
            )

            if 1 <= page_num <= max_page:

                CURRENT_PAGE = page_num - 1

                render_catalog()

        except:

            print("⚠️ PAGE ERROR")

        return

    # DELETE NAME
    if low.startswith('delete "'):

        try:

            model_name = re.findall(

                r'delete\s+"(.+)"',

                choice,

                flags=re.I

            )[0]

            delete_model(model_name)

            refresh_all_data()

            render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    # DELETE ID
    if low.startswith("delete "):

        try:

            delete_id = int(
                choice.split()[1]
            )

            idx = delete_id - 101

            if 0 <= idx < len(STATE["installed"]):

                target = STATE["installed"][idx]

                delete_model(target)

                refresh_all_data()

                render_catalog()

        except:

            print("⚠️ DELETE ERROR")

        return

    # INFO
    if low.startswith("info "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            idx = int(arg) - 1

            if 0 <= idx < len(STATE["library"]):

                show_model_info(
                    STATE["library"][idx]
                )

        else:

            show_model_info(arg)

        return

    # REFRESH
    if low == "refresh":

        refresh_all_data()

        render_catalog()

        return

    # STOP NGROK
    if low == "stop ngrok":

        stop_ngrok()

        return

    # STOP SERVER
    if low == "stop server":

        stop_ollama_server()

        return

    # NGROK
    if low == "0":

        run_ngrok()

        return

    # PULL BY ID
    if choice.isdigit():

        idx = int(choice) - 1

        if 0 <= idx < len(STATE["library"]):

            target = STATE["library"][idx]

            pull_model(target)

            refresh_all_data()

            render_catalog()

        return

    # MANUAL PULL
    pull_model(choice)

    refresh_all_data()

    render_catalog()

# ==============================================================================
# RENDER
# ==============================================================================

def render_catalog():

    with OUTPUT:

        clear_output(wait=True)

        total_models = len(
            STATE["library"]
        )

        max_page = max(
            1,
            (total_models - 1)
            // MODELS_PER_PAGE + 1
        )

        start = CURRENT_PAGE * MODELS_PER_PAGE

        end = start + MODELS_PER_PAGE

        page_models = STATE["library"][start:end]

        print("═" * 150)

        print(
            "ID   | SRC | STATUS         | MODEL                              | SIZE     | QUANT    | DESCRIPTION"
        )

        print("─" * 150)

        installed = set(
            STATE["installed"]
        )

        for offset, slug in enumerate(
            page_models
        ):

            i = start + offset + 1

            # SOURCE
            if slug.startswith("hf.co/"):
                source = "🤗"

            elif slug.startswith("lmstudio/"):
                source = "🧠"

            else:
                source = "🦙"

            # STATUS
            if slug in installed:
                status = "✅ INSTALLED"
            else:
                status = "⚪ EMPTY"

            detail = STATE["details"].get(
                slug,
                {}
            )

            quant = detail.get(
                "quant",
                "---"
            )

            size = detail.get(
                "size",
                "---"
            )

            desc = detail.get(
                "reason",
                ""
            )

            display_slug = slug[:34]

            print(

                f"{str(i):<4} | "

                f"{source:<4} | "

                f"{status:<14} | "

                f"{display_slug:<34} | "

                f"{size:<8} | "

                f"{quant:<8} | "

                f"{desc}"

            )

        print("\n📂 INSTALLED MODELS:")

        for i, name in enumerate(

            STATE["installed"],

            101

        ):

            print(
                f"{i:<4} | 🗑️ {name}"
            )

        print("\n" + "═" * 150)

        ollama_count = len([
            x for x in STATE["library"]
            if not x.startswith("hf.co/")
            and not x.startswith("lmstudio/")
        ])

        lmstudio_count = len([
            x for x in STATE["library"]
            if x.startswith("lmstudio/")
        ])

        hf_count = len([
            x for x in STATE["library"]
            if x.startswith("hf.co/")
        ])

        print(
            f"🦙 Ollama: {ollama_count} | "
            f"🧠 LM Studio: {lmstudio_count} | "
            f"🤗 HF: {hf_count}"
        )

        print(
            f"\n📄 PAGE: "
            f"{CURRENT_PAGE + 1}/{max_page}"
        )

        print("\nCOMMANDS:")

        print("1                    -> pull model")
        print("delete 101           -> delete by id")
        print('delete "llama3"      -> delete by name')
        print("info 1               -> model info")
        print("page 5               -> jump page")
        print("refresh              -> refresh")
        print("stop ngrok           -> stop ngrok")
        print("stop server          -> stop ollama")
        print("0                    -> start ngrok")

# ==============================================================================
# SUBMIT
# ==============================================================================

def on_submit(_=None):

    cmd = INPUT_BOX.value.strip()

    INPUT_BOX.value = ""

    if not cmd:
        return

    with OUTPUT:

        print(f"\n👉 {cmd}")

    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================

restart_ollama_server()

refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================

TITLE = widgets.HTML("""
<h2>🚀 Ollama Colab Manager</h2>
""")

HELP = widgets.HTML("""
<b>
Commands:
1 | delete 101 | info 1 | stop ngrok | stop server
</b>
""")

INPUT_BOX = widgets.Text(

    placeholder="Nhập lệnh...",

    description=">>>",

    layout=widgets.Layout(width="90%")

)

SEND_BTN = widgets.Button(

    description="Gửi",

    button_style="success"

)

REFRESH_BTN = widgets.Button(

    description="Refresh",

    button_style="info"

)

NGROK_BTN = widgets.Button(

    description="Start Ngrok",

    button_style="warning"

)

STOP_NGROK_BTN = widgets.Button(

    description="Stop Ngrok",

    button_style="danger"

)

STOP_SERVER_BTN = widgets.Button(

    description="Stop Server",

    button_style="danger"

)

OUTPUT = widgets.Output()

# ==============================================================================
# EVENTS
# ==============================================================================

SEND_BTN.on_click(on_submit)

INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(

    lambda _: (
        refresh_all_data(),
        render_catalog()
    )

)

NGROK_BTN.on_click(
    lambda _: run_ngrok()
)

STOP_NGROK_BTN.on_click(
    lambda _: stop_ngrok()
)

STOP_SERVER_BTN.on_click(
    lambda _: stop_ollama_server()
)

# ==============================================================================
# BUTTONS
# ==============================================================================

BUTTONS = widgets.HBox([

    SEND_BTN,

    REFRESH_BTN,

    NGROK_BTN,

    STOP_NGROK_BTN,

    STOP_SERVER_BTN

])

# ==============================================================================
# MAIN UI
# ==============================================================================

UI = widgets.VBox([

    TITLE,

    OUTPUT,

    HELP,

    INPUT_BOX,

    BUTTONS

])

display(UI)

render_catalog()

7. ollama + hugging face

In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - ENHANCED VERIFIED EDITION
# ==============================================================================
# FEATURES:
# - Ollama library crawler
# - HuggingFace GGUF crawler
# - Compatibility verification
# - Pagination
# - Realtime pull progress
# - Delete by ID or name
# - Info command with model ID
# - Ngrok controls
# - Ollama server controls
# - Better UI rendering
# - Non-breaking upgrade
# ==============================================================================

import os
import re
import time
import subprocess
from urllib.parse import unquote, quote
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from IPython.display import display, clear_output

# ==============================================================================
# WIDGETS
# ==============================================================================
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except:
    WIDGETS_AVAILABLE = False

# ==============================================================================
# CONFIG
# ==============================================================================
OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_PORT = "11434"

MAX_LIBRARY_PAGES = 20
MAX_DETAIL_WORKERS = 12
MAX_DETAIL_FETCH = 300
HTTP_TIMEOUT = 20

HF_MAX_MODELS = 700

VERIFY_HF_MODELS = True
SKIP_SHARDED_MODELS = False

MODELS_PER_PAGE = 35
CURRENT_PAGE = 0

# ==============================================================================
# MODEL DB
# ==============================================================================
MODEL_DB = {
    "llama3.1": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Đa năng, hiểu tiếng Việt cực tốt."
    },

    "llama3.2": {
        "s": "2.0GB",
        "c": "128K",
        "n": "Nhỏ gọn, tối ưu cho thiết bị yếu."
    },

    "deepseek-r1": {
        "s": "4.7GB+",
        "c": "128K",
        "n": "Suy luận/code mạnh."
    },

    "qwen2.5-coder": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Tối ưu lập trình."
    },

    "phi4": {
        "s": "8GB",
        "c": "128K",
        "n": "Logic mạnh."
    },

    "gemma2": {
        "s": "5.5GB",
        "c": "8K",
        "n": "Google model."
    },

    "mistral": {
        "s": "4.1GB",
        "c": "32K",
        "n": "Chatbot ổn định."
    },

    "llava": {
        "s": "4.5GB",
        "c": "Vision",
        "n": "Nhìn ảnh."
    },
}

# ==============================================================================
# GLOBAL STATE
# ==============================================================================
STATE = {
    "installed": [],
    "library": [],
    "details": {},
}

# ==============================================================================
# SHELL
# ==============================================================================
def run_cmd(args, timeout=60):

    try:

        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return (
            p.returncode,
            p.stdout.strip(),
            p.stderr.strip()
        )

    except Exception as e:

        return (
            1,
            "",
            str(e)
        )

# ==============================================================================
# CURL
# ==============================================================================
def curl_get(url):

    code, out, err = run_cmd([
        "curl",
        "-L",
        "-sS",
        "--max-time",
        str(HTTP_TIMEOUT),
        url
    ])

    if code != 0:
        return ""

    return out

# ==============================================================================
# SERVER
# ==============================================================================
def restart_ollama_server():

    print("🔄 Đang khởi động Ollama server...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    os.environ["OLLAMA_HOST"] = OLLAMA_HOST

    log_file = open("ollama.log", "w")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=log_file
    )

    time.sleep(6)

    print("✅ Ollama server sẵn sàng!")

def stop_ollama_server():

    print("\n🛑 Đang tắt Ollama server...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    print("✅ Đã tắt Ollama server!")

# ==============================================================================
# NGROK
# ==============================================================================
def run_ngrok():

    print("\n🚀 Đang khởi chạy ngrok...")

    run_cmd([
        "bash",
        "-lc",
        f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"
    ])

    time.sleep(3)

    try:

        response = requests.get(
            "http://localhost:4040/api/tunnels",
            timeout=5
        )

        tunnels = response.json().get("tunnels", [])

        if tunnels:

            url = tunnels[0].get("public_url")

            print(f"\n✅ NGROK ONLINE:")
            print(url)

        else:

            print("⚠️ Không tìm thấy tunnel.")

    except:

        print("⚠️ Không lấy được link ngrok!")

def stop_ngrok():

    print("\n🛑 Đang tắt ngrok...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ngrok || true"
    ])

    print("✅ Đã tắt ngrok!")

# ==============================================================================
# INSTALLED MODELS
# ==============================================================================
def get_installed_models():

    code, out, err = run_cmd(
        ["ollama", "list"],
        timeout=20
    )

    if code != 0:
        return []

    lines = out.splitlines()

    if len(lines) <= 1:
        return []

    models = []

    for line in lines[1:]:

        parts = line.split()

        if not parts:
            continue

        name = parts[0].split(":")[0]

        models.append(name)

    return models

# ==============================================================================
# FETCH OLLAMA LIBRARY
# ==============================================================================
def fetch_library_models():

    models = []

    for page in range(1, MAX_LIBRARY_PAGES + 1):

        if page == 1:
            url = "https://ollama.com/library"
        else:
            url = f"https://ollama.com/library?page={page}"

        html = curl_get(url)

        if not html:
            break

        found = re.findall(
            r'href="/library/([^"]+)"',
            html
        )

        cleaned = []

        for item in found:

            item = unquote(item)

            if "/" in item:
                continue

            if item not in cleaned:
                cleaned.append(item)

        new_items = [
            x for x in cleaned
            if x not in models
        ]

        if not new_items and page > 1:
            break

        models.extend(new_items)

    return models

# ==============================================================================
# FETCH HUGGINGFACE GGUF
# ==============================================================================
def fetch_huggingface_models():

    models = []

    try:

        url = (
            "https://huggingface.co/api/models"
            "?search=GGUF"
            # "&limit=1000"
            "&full=true"
        )

        response = requests.get(
            url,
            timeout=40
        )

        data = response.json()

        for item in data:

            model_id = item.get("id", "")

            if not model_id:
                continue

            siblings = item.get("siblings", [])

            gguf_files = []

            for s in siblings:

                fname = s.get(
                    "rfilename",
                    ""
                ).lower()

                if fname.endswith(".gguf"):
                    gguf_files.append(fname)

            if not gguf_files:
                continue

            is_sharded = any(
                "-00001-of-" in x
                for x in gguf_files
            )

            if SKIP_SHARDED_MODELS and is_sharded:
                continue

            compatible = False
            compat_reason = "Unknown"

            if gguf_files and not is_sharded:

                compatible = True
                compat_reason = "Verified GGUF"

            elif is_sharded:

                compat_reason = "Sharded GGUF"

            quant = "---"

            quant_patterns = [
                "q2_k",
                "q3_k",
                "q4_0",
                "q4_k_m",
                "q5_k_m",
                "q6_k",
                "q8_0",
                "iq1",
                "iq2",
                "iq3",
                "iq4"
            ]

            for q in quant_patterns:

                if any(q in x for x in gguf_files):

                    quant = q.upper()

                    break

            model_name = f"hf.co/{model_id}"

            models.append({
                "name": model_name,
                "compatible": compatible,
                "reason": compat_reason,
                "quant": quant
            })

            if len(models) >= HF_MAX_MODELS:
                break

    except Exception as e:

        print(f"⚠️ HF FETCH ERROR: {e}")

    return models

# ==============================================================================
# DETAIL
# ==============================================================================
def fetch_model_detail(slug):

    result = {
        "slug": slug,
        "description": "",
        "size": "",
        "context": "",
    }

    try:

        if slug.startswith("hf.co/"):

            result["description"] = (
                "HuggingFace GGUF Model"
            )

            return result

        html = curl_get(
            f"https://ollama.com/library/{quote(slug)}"
        )

        if html:

            desc = re.search(
                r'<meta name="description" content="([^"]+)"',
                html
            )

            if desc:
                result["description"] = desc.group(1)

        tags_html = curl_get(
            f"https://ollama.com/library/{quote(slug)}/tags"
        )

        if tags_html:

            text = re.sub(
                r"<[^>]+>",
                " ",
                tags_html
            )

            size_match = re.search(
                r'([0-9.]+\s?(?:GB|MB))',
                text
            )

            ctx_match = re.search(
                r'([0-9.]+\s?[KMG]?)\s*context',
                text,
                flags=re.I
            )

            if size_match:
                result["size"] = size_match.group(1)

            if ctx_match:
                result["context"] = ctx_match.group(1)

    except:
        pass

    return result

# ==============================================================================
# REFRESH
# ==============================================================================
def refresh_all_data():

    print("🔄 Đang tải danh sách models...")

    STATE["installed"] = get_installed_models()

    ollama_models = fetch_library_models()

    hf_models = fetch_huggingface_models()

    merged = []

    for m in ollama_models:

        if m not in merged:
            merged.append(m)

    for item in hf_models:

        model_name = item["name"]

        if model_name not in merged:
            merged.append(model_name)

        STATE["details"][model_name] = {
            "compat": item["compatible"],
            "reason": item["reason"],
            "quant": item["quant"]
        }

    merged = sorted(
        merged,
        key=lambda x: (
            x.startswith("hf.co/"),
            x.lower()
        )
    )

    STATE["library"] = merged

    details = {}

    targets = STATE["library"][:MAX_DETAIL_FETCH]

    with ThreadPoolExecutor(
        max_workers=MAX_DETAIL_WORKERS
    ) as ex:

        futures = {
            ex.submit(fetch_model_detail, slug): slug
            for slug in targets
        }

        for fut in as_completed(futures):

            slug = futures[fut]

            try:
                details[slug] = fut.result()
            except:
                pass

    for k, v in STATE["details"].items():

        if k not in details:
            details[k] = {}

        details[k].update(v)

    STATE["details"] = details

# ==============================================================================
# PULL MODEL
# ==============================================================================
def pull_model(model_name):

    print("\n" + "═" * 120)

    print(f"📥 ĐANG TẢI MODEL: {model_name}")

    print("═" * 120)

    try:

        process = subprocess.Popen(
            ["ollama", "pull", model_name],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        last_line = ""

        while True:

            line = process.stdout.readline()

            if not line and process.poll() is not None:
                break

            if not line:
                continue

            line = line.strip()

            if not line:
                continue

            pretty = line

            pretty = pretty.replace(
                "pulling manifest",
                "📦 pulling manifest"
            )

            pretty = pretty.replace(
                "verifying sha256 digest",
                "🔐 verifying digest"
            )

            pretty = pretty.replace(
                "writing manifest",
                "💾 writing manifest"
            )

            pretty = pretty.replace(
                "removing any unused layers",
                "🧹 cleaning layers"
            )

            pretty = pretty.replace(
                "success",
                "✅ success"
            )

            if "%" in pretty:
                pretty = "⬇️  " + pretty

            if pretty != last_line:

                print(f"\033[1;36m{pretty}\033[0m")

                last_line = pretty

        rc = process.poll()

        print("\n" + "═" * 120)

        if rc == 0:
            print(f"✅ TẢI MODEL '{model_name}' THÀNH CÔNG!")
        else:
            print(f"⚠️ TẢI MODEL '{model_name}' THẤT BẠI!")

        print("═" * 120)

    except Exception as e:

        print(f"❌ LỖI: {e}")

# ==============================================================================
# DELETE MODEL
# ==============================================================================
def delete_model(model_name):

    print(f"\n🗑️ Đang xóa: {model_name}")

    run_cmd([
        "ollama",
        "rm",
        model_name
    ])

    print("✅ Xóa hoàn tất!")

# ==============================================================================
# INFO
# ==============================================================================
def show_model_info(model_name):

    detail = STATE["details"].get(model_name)

    if not detail:
        detail = fetch_model_detail(model_name)

    try:
        model_id = STATE["library"].index(model_name) + 1
    except:
        model_id = "?"

    print("\n" + "═" * 120)

    print(f"ID: {model_id}")
    print(f"MODEL: {model_name}")

    print("═" * 120)

    print(f"SIZE: {detail.get('size')}")
    print(f"CONTEXT: {detail.get('context')}")
    print(f"QUANT: {detail.get('quant')}")
    print(f"COMPAT: {detail.get('reason')}")
    print(f"DESC: {detail.get('description')}")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================
def process_command(choice):

    global CURRENT_PAGE

    low = choice.lower()

    # next
    if low == "next":

        max_page = max(
            0,
            (len(STATE["library"]) - 1) // MODELS_PER_PAGE
        )

        if CURRENT_PAGE < max_page:
            CURRENT_PAGE += 1

        render_catalog()

        return

    # prev
    if low == "prev":

        if CURRENT_PAGE > 0:
            CURRENT_PAGE -= 1

        render_catalog()

        return

    # refresh
    if low == "refresh":

        refresh_all_data()

        render_catalog()

        return

    # start ngrok
    if low == "0":

        run_ngrok()

        return

    # stop ngrok
    if low == "stop ngrok":

        stop_ngrok()

        return

    # stop server
    if low == "stop server":

        stop_ollama_server()

        return

    # info
    if low.startswith("info "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            idx = int(arg) - 1

            if 0 <= idx < len(STATE["library"]):

                show_model_info(
                    STATE["library"][idx]
                )

        else:

            show_model_info(arg)

        return

    # delete
    if low.startswith("delete "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            delete_id = int(arg)

            idx = delete_id - 101

            if 0 <= idx < len(STATE["installed"]):

                target = STATE["installed"][idx]

                delete_model(target)

        else:

            delete_model(arg)

        refresh_all_data()

        render_catalog()

        return

    # pull by id
    if choice.isdigit():

        idx = int(choice) - 1

        if 0 <= idx < len(STATE["library"]):

            target = STATE["library"][idx]

            pull_model(target)

            refresh_all_data()

            render_catalog()

        return

    # manual pull
    pull_model(choice)

    refresh_all_data()

    render_catalog()

# ==============================================================================
# RENDER
# ==============================================================================
def render_catalog():

    with OUTPUT:

        clear_output(wait=True)

        print("═" * 170)

        print(
            "ID   | SRC | STATUS         | MODEL                              | QUANT    | SIZE      | CTX       | DESCRIPTION"
        )

        print("─" * 170)

        installed = set(
            STATE["installed"]
        )

        start_idx = CURRENT_PAGE * MODELS_PER_PAGE
        end_idx = start_idx + MODELS_PER_PAGE

        visible_models = STATE["library"][start_idx:end_idx]

        for i, slug in enumerate(
            visible_models,
            start_idx + 1
        ):

            if slug in installed:
                status = "✅ INSTALLED"
            else:
                status = "⚪ EMPTY"

            detail = STATE["details"].get(
                slug,
                {}
            )

            compat = detail.get("compat")
            quant = detail.get("quant", "---")

            if slug.startswith("hf.co/"):

                if compat is True:
                    source = "🤗🟢"

                elif compat is False:
                    source = "🤗🔴"

                else:
                    source = "🤗"

            else:

                source = "🦙"

            fallback = MODEL_DB.get(
                slug.lower(),
                {}
            )

            size = (
                detail.get("size")
                or fallback.get("s", "---")
            )

            ctx = (
                detail.get("context")
                or fallback.get("c", "---")
            )

            desc = (
                detail.get("description")
                or fallback.get("n", "")
            )

            if len(desc) > 45:
                desc = desc[:45] + "..."

            display_slug = slug

            if len(display_slug) > 34:
                display_slug = display_slug[:31] + "..."

            print(
                f"{str(i):<4} | "
                f"{source:<4} | "
                f"{status:<14} | "
                f"{display_slug:<34} | "
                f"{quant:<8} | "
                f"{size:<9} | "
                f"{ctx:<9} | "
                f"{desc}"
            )

        print("\n📂 MODEL ĐÃ CÀI:")

        for i, name in enumerate(
            STATE["installed"],
            101
        ):

            print(
                f"{i:<4} | 🗑️ {name}"
            )

        max_page = max(
            1,
            (len(STATE["library"]) - 1) // MODELS_PER_PAGE + 1
        )

        print(f"\n📄 PAGE: {CURRENT_PAGE + 1}/{max_page}")

        print(f"📦 TOTAL MODELS: {len(STATE['library'])}")

        print("\n═" * 170)

        print("COMMANDS:")
        print("1                     -> tải model")
        print("delete 101            -> xóa model theo ID")
        print("delete llama3.2       -> xóa model theo tên")
        print("info 1                -> xem thông tin")
        print("next                  -> trang tiếp")
        print("prev                  -> trang trước")
        print("refresh               -> reload library")
        print("stop ngrok            -> tắt ngrok")
        print("stop server           -> tắt ollama")
        print("0                     -> start ngrok")

# ==============================================================================
# SUBMIT
# ==============================================================================
def on_submit(_=None):

    cmd = INPUT_BOX.value.strip()

    INPUT_BOX.value = ""

    if not cmd:
        return

    with OUTPUT:

        print(f"\n👉 {cmd}")

    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================
restart_ollama_server()

refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================
TITLE = widgets.HTML("""
<h2>🚀 Ollama Colab Manager</h2>
""")

HELP = widgets.HTML("""
<b>
Commands:
1 | delete 101 | delete llama3.2 | info 1
</b>
""")

INPUT_BOX = widgets.Text(
    placeholder="Nhập lệnh...",
    description=">>>",
    layout=widgets.Layout(width="90%")
)

SEND_BTN = widgets.Button(
    description="Gửi",
    button_style="success"
)

REFRESH_BTN = widgets.Button(
    description="Refresh",
    button_style="info"
)

NGROK_BTN = widgets.Button(
    description="Start Ngrok",
    button_style="warning"
)

STOP_NGROK_BTN = widgets.Button(
    description="Stop Ngrok",
    button_style="danger"
)

STOP_SERVER_BTN = widgets.Button(
    description="Stop Server",
    button_style="danger"
)

OUTPUT = widgets.Output()

# ==============================================================================
# EVENTS
# ==============================================================================
SEND_BTN.on_click(on_submit)

INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(
    lambda _: (
        refresh_all_data(),
        render_catalog()
    )
)

NGROK_BTN.on_click(
    lambda _: run_ngrok()
)

STOP_NGROK_BTN.on_click(
    lambda _: stop_ngrok()
)

STOP_SERVER_BTN.on_click(
    lambda _: stop_ollama_server()
)

# ==============================================================================
# BUTTONS
# ==============================================================================
BUTTONS = widgets.HBox([

    SEND_BTN,

    REFRESH_BTN,

    NGROK_BTN,

    STOP_NGROK_BTN,

    STOP_SERVER_BTN

])

# ==============================================================================
# MAIN UI
# ==============================================================================
UI = widgets.VBox([

    TITLE,

    OUTPUT,

    HELP,

    INPUT_BOX,

    BUTTONS

])

display(UI)

render_catalog()

ollama lib only

In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - FULL REBUILD STABLE VERSION
# ==============================================================================
# FEATURES:
# - Full Ollama library fetch
# - Playwright dynamic scrolling
# - Metadata fetch
# - Realtime pull progress
# - Ngrok controls
# - Delete models
# - Info command
# - Stable Colab widgets UI
# ==============================================================================

# ==============================================================================
# INSTALL (RUN ONCE IN COLAB)
# ==============================================================================
# !pip -q install playwright ipywidgets
# !playwright install chromium

import os
import re
import time
import subprocess
from urllib.parse import unquote, quote
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

from IPython.display import display, clear_output

from playwright.sync_api import sync_playwright

import ipywidgets as widgets

# ==============================================================================
# CONFIG
# ==============================================================================
OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_PORT = "11434"

MAX_LIBRARY_PAGES = 25
MAX_DETAIL_FETCH = 300
MAX_DETAIL_WORKERS = 10

HTTP_TIMEOUT = 20

# ==============================================================================
# MODEL DB
# ==============================================================================
MODEL_DB = {

    "llama3.1": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Đa năng."
    },

    "llama3.2": {
        "s": "2GB",
        "c": "128K",
        "n": "Nhẹ."
    },

    "deepseek-r1": {
        "s": "4.7GB+",
        "c": "128K",
        "n": "Code mạnh."
    },

    "qwen2.5-coder": {
        "s": "4.7GB",
        "c": "128K",
        "n": "Coder."
    },

    "phi4": {
        "s": "8GB",
        "c": "128K",
        "n": "Logic."
    },

    "mistral": {
        "s": "4.1GB",
        "c": "32K",
        "n": "Chat."
    },

    "llava": {
        "s": "4.5GB",
        "c": "Vision",
        "n": "Image."
    }
}

# ==============================================================================
# GLOBAL STATE
# ==============================================================================
STATE = {
    "installed": [],
    "library": [],
    "details": {},
}

# ==============================================================================
# OUTPUT
# ==============================================================================
OUTPUT = widgets.Output()

# ==============================================================================
# SHELL
# ==============================================================================
def run_cmd(args, timeout=60):

    try:

        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        return (
            p.returncode,
            p.stdout.strip(),
            p.stderr.strip()
        )

    except Exception as e:

        return (
            1,
            "",
            str(e)
        )

# ==============================================================================
# CURL
# ==============================================================================
def curl_get(url):

    code, out, err = run_cmd([
        "curl",
        "-L",
        "-sS",
        "--max-time",
        str(HTTP_TIMEOUT),
        url
    ])

    if code != 0:
        return ""

    return out

# ==============================================================================
# SERVER
# ==============================================================================
def restart_ollama_server():

    with OUTPUT:

        print("🔄 Starting Ollama server...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    os.environ["OLLAMA_HOST"] = OLLAMA_HOST

    log_file = open("ollama.log", "w")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=log_file
    )

    time.sleep(6)

    with OUTPUT:

        print("✅ Ollama server ready!")

# ==============================================================================
# STOP SERVER
# ==============================================================================
def stop_ollama_server():

    with OUTPUT:

        print("\n🛑 Stopping Ollama server...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ollama || true"
    ])

    with OUTPUT:

        print("✅ Ollama server stopped!")

# ==============================================================================
# NGROK
# ==============================================================================
def run_ngrok():

    with OUTPUT:

        print("\n🚀 Starting ngrok...")

    run_cmd([
        "bash",
        "-lc",
        f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"
    ])

    time.sleep(3)

    try:

        response = requests.get(
            "http://localhost:4040/api/tunnels",
            timeout=5
        )

        tunnels = response.json().get("tunnels", [])

        if tunnels:

            url = tunnels[0]["public_url"]

            with OUTPUT:

                print("\n✅ NGROK ONLINE:")
                print(url)

        else:

            with OUTPUT:

                print("⚠️ No tunnel.")

    except:

        with OUTPUT:

            print("⚠️ Cannot get ngrok URL.")

# ==============================================================================
# STOP NGROK
# ==============================================================================
def stop_ngrok():

    with OUTPUT:

        print("\n🛑 Stopping ngrok...")

    run_cmd([
        "bash",
        "-lc",
        "pkill ngrok || true"
    ])

    with OUTPUT:

        print("✅ Ngrok stopped!")

# ==============================================================================
# INSTALLED
# ==============================================================================
def get_installed_models():

    code, out, err = run_cmd(
        ["ollama", "list"],
        timeout=20
    )

    if code != 0:
        return []

    lines = out.splitlines()

    if len(lines) <= 1:
        return []

    models = []

    for line in lines[1:]:

        parts = line.split()

        if not parts:
            continue

        name = parts[0].split(":")[0]

        models.append(name)

    return models

# ==============================================================================
# PARSE HTML
# ==============================================================================
def parse_html_for_slugs(html):

    found = re.findall(
        r'href="/library/([^"]+)"',
        html,
        flags=re.I
    )

    result = []

    for item in found:

        item = unquote(item).strip()

        if not item:
            continue

        if "/" in item:
            continue

        low = item.lower()

        if low in ["library", "search", "tags"]:
            continue

        result.append(item)

    return result

# ==============================================================================
# FETCH LIBRARY
# ==============================================================================
# ==============================================================================
# FETCH LIBRARY MODELS - ULTRA DISCOVERY VERSION
# ==============================================================================
def fetch_library_models():

    models = set()

    # ==========================================================================
    # PAGE SCRAPE
    # ==========================================================================
    with OUTPUT:

        print("🌐 Fetching library pages...")

    try:

        for page in range(1, MAX_LIBRARY_PAGES + 1):

            if page == 1:
                url = "https://ollama.com/library"
            else:
                url = f"https://ollama.com/library?page={page}"

            html = curl_get(url)

            if not html:
                continue

            found = parse_html_for_slugs(html)

            before = len(models)

            for item in found:
                models.add(item)

            with OUTPUT:

                clear_output(wait=True)

                print(f"🌐 PAGE {page}")
                print(f"📦 MODELS: {len(models)}")

            if len(models) == before and page > 3:
                break

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ PAGE FETCH ERROR: {e}")

    # ==========================================================================
    # SEARCH DISCOVERY
    # ==========================================================================
    with OUTPUT:

        print("\n🔎 Search discovery...")

    keywords = [

        "a","b","c","d","e","f","g","h","i","j",
        "k","l","m","n","o","p","q","r","s","t",
        "u","v","w","x","y","z",

        "llama",
        "deepseek",
        "qwen",
        "coder",
        "vision",
        "embed",
        "math",
        "phi",
        "gemma",
        "mistral",
        "tiny",
        "large",
        "small",
        "moe",
        "reason",
        "chat",
        "tool",
        "sql",
        "agent",
        "audio",
        "image",
        "translate",
        "coding",
        "medical",
        "roleplay",
        "story",
        "anime",
        "uncensored",

    ]

    try:

        for kw in keywords:

            url = f"https://ollama.com/search?q={quote(kw)}"

            html = curl_get(url)

            if not html:
                continue

            found = parse_html_for_slugs(html)

            for item in found:
                models.add(item)

            with OUTPUT:

                clear_output(wait=True)

                print(f"🔎 SEARCH: {kw}")
                print(f"📦 MODELS: {len(models)}")

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ SEARCH ERROR: {e}")

    # ==========================================================================
    # CATEGORY DISCOVERY
    # ==========================================================================
    with OUTPUT:

        print("\n🧠 Category discovery...")

    categories = [

        "vision",
        "embedding",
        "thinking",
        "tools",
        "featured",
        "code",
        "chat",

    ]

    try:

        for cat in categories:

            url = f"https://ollama.com/search?c={cat}"

            html = curl_get(url)

            if not html:
                continue

            found = parse_html_for_slugs(html)

            for item in found:
                models.add(item)

            with OUTPUT:

                clear_output(wait=True)

                print(f"🧠 CATEGORY: {cat}")
                print(f"📦 MODELS: {len(models)}")

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ CATEGORY ERROR: {e}")

    # ==========================================================================
    # PLAYWRIGHT SCROLL
    # ==========================================================================
    try:

        with OUTPUT:

            print("\n🚀 Playwright deep scan...")

        with sync_playwright() as p:

            browser = p.chromium.launch(
                headless=True
            )

            page = browser.new_page()

            page.goto(
                "https://ollama.com/library",
                wait_until="domcontentloaded",
                timeout=120000
            )

            time.sleep(3)

            stable = 0
            last_count = 0

            for step in range(120):

                page.evaluate(
                    "window.scrollTo(0, document.body.scrollHeight)"
                )

                time.sleep(1.2)

                html = page.content()

                found = parse_html_for_slugs(html)

                for item in found:
                    models.add(item)

                current = len(models)

                with OUTPUT:

                    clear_output(wait=True)

                    print(f"🚀 PLAYWRIGHT STEP: {step}")
                    print(f"📦 MODELS: {current}")

                if current == last_count:
                    stable += 1
                else:
                    stable = 0

                last_count = current

                if stable >= 6:
                    break

            browser.close()

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ PLAYWRIGHT ERROR: {e}")

    # ==========================================================================
    # SITEMAP
    # ==========================================================================
    try:

        with OUTPUT:

            print("\n🗺️ Reading sitemap...")

        sitemap = curl_get(
            "https://ollama.com/sitemap.xml"
        )

        found = re.findall(
            r'https://ollama\.com/library/([^<"\s]+)',
            sitemap
        )

        for item in found:

            item = unquote(item).strip()

            if "/" in item:
                continue

            models.add(item)

        with OUTPUT:

            print(f"📦 MODELS AFTER SITEMAP: {len(models)}")

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ SITEMAP ERROR: {e}")

    # ==========================================================================
    # COMMON SLUG BRUTEFORCE
    # ==========================================================================
    common_slugs = [

        "deepseek-v2",
        "deepseek-coder",
        "deepseek-r1",
        "deepseek-v3",
        "qwen2.5",
        "qwen2.5-coder",
        "qwen2.5-vl",
        "qwen3",
        "qwen3-coder",
        "llama3",
        "llama3.1",
        "llama3.2",
        "llama3.3",
        "codellama",
        "tinyllama",
        "wizardcoder",
        "wizardlm",
        "phi3",
        "phi4",
        "mistral",
        "mixtral",
        "gemma2",
        "gemma3",
        "openchat",
        "solar",
        "yi",
        "command-r",
        "command-r-plus",
        "dolphin-mistral",
        "nous-hermes2",
        "stable-beluga",
        "medllama2",
        "orca2",

    ]

    with OUTPUT:

        print("\n🧪 Bruteforce validating common models...")

    try:

        for slug in common_slugs:

            if slug in models:
                continue

            html = curl_get(
                f"https://ollama.com/library/{slug}"
            )

            if html and "Ollama" in html:
                models.add(slug)

            with OUTPUT:

                clear_output(wait=True)

                print(f"🧪 VALIDATING: {slug}")
                print(f"📦 MODELS: {len(models)}")

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ BRUTEFORCE ERROR: {e}")

    # ==========================================================================
    # CLEAN
    # ==========================================================================
    cleaned = []

    for item in models:

        item = item.strip()

        if not item:
            continue

        if "/" in item:
            continue

        if item not in cleaned:
            cleaned.append(item)

    cleaned = sorted(cleaned)

    with OUTPUT:

        clear_output(wait=True)

        print(f"✅ TOTAL MODELS FETCHED: {len(cleaned)}")

    return cleaned
# ==============================================================================
# DETAIL
# ==============================================================================
def fetch_model_detail(slug):

    result = {
        "slug": slug,
        "description": "",
        "size": "",
        "context": "",
    }

    try:

        html = curl_get(
            f"https://ollama.com/library/{quote(slug)}"
        )

        desc = re.search(
            r'<meta name="description" content="([^"]+)"',
            html
        )

        if desc:
            result["description"] = desc.group(1)

        tags_html = curl_get(
            f"https://ollama.com/library/{quote(slug)}/tags"
        )

        text = re.sub(
            r"<[^>]+>",
            " ",
            tags_html
        )

        size_match = re.search(
            r'([0-9.]+\s?(?:GB|MB))',
            text
        )

        ctx_match = re.search(
            r'([0-9.]+\s?[KMG]?)\s*context',
            text,
            flags=re.I
        )

        if size_match:
            result["size"] = size_match.group(1)

        if ctx_match:
            result["context"] = ctx_match.group(1)

    except:
        pass

    return result

# ==============================================================================
# REFRESH
# ==============================================================================
def refresh_all_data():

    STATE["installed"] = get_installed_models()

    try:

        STATE["library"] = fetch_library_models()

    except Exception as e:

        with OUTPUT:

            print(f"⚠️ Fetch error: {e}")

        STATE["library"] = []

    details = {}

    targets = STATE["library"][:MAX_DETAIL_FETCH]

    with OUTPUT:

        print("\n🧠 Fetching metadata...\n")

    with ThreadPoolExecutor(
        max_workers=MAX_DETAIL_WORKERS
    ) as ex:

        futures = {
            ex.submit(fetch_model_detail, slug): slug
            for slug in targets
        }

        total = len(futures)
        done_count = 0

        for fut in as_completed(futures):

            slug = futures[fut]

            try:
                details[slug] = fut.result()
            except:
                details[slug] = {}

            done_count += 1

            with OUTPUT:

                print(
                    f"📦 Metadata: {done_count}/{total}",
                    end="\r"
                )

    STATE["details"] = details

# ==============================================================================
# PULL MODEL
# ==============================================================================
def pull_model(model_name):

    with OUTPUT:

        print("\n" + "═" * 120)

        print(f"📥 DOWNLOADING: {model_name}")

        print("═" * 120)

    try:

        process = subprocess.Popen(
            ["ollama", "pull", model_name],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        last_line = ""

        while True:

            line = process.stdout.readline()

            if not line and process.poll() is not None:
                break

            if not line:
                continue

            line = line.strip()

            if not line:
                continue

            pretty = line

            pretty = pretty.replace(
                "pulling manifest",
                "📦 pulling manifest"
            )

            pretty = pretty.replace(
                "verifying sha256 digest",
                "🔐 verifying digest"
            )

            pretty = pretty.replace(
                "writing manifest",
                "💾 writing manifest"
            )

            pretty = pretty.replace(
                "removing any unused layers",
                "🧹 cleaning layers"
            )

            pretty = pretty.replace(
                "success",
                "✅ success"
            )

            if "%" in pretty:
                pretty = "⬇️  " + pretty

            if pretty != last_line:

                with OUTPUT:

                    print(pretty)

                last_line = pretty

        rc = process.poll()

        with OUTPUT:

            print("\n" + "═" * 120)

            if rc == 0:
                print(f"✅ INSTALLED: {model_name}")
            else:
                print(f"⚠️ FAILED: {model_name}")

            print("═" * 120)

    except Exception as e:

        with OUTPUT:

            print(f"❌ ERROR: {e}")

# ==============================================================================
# DELETE
# ==============================================================================
def delete_model(model_name):

    with OUTPUT:

        print(f"\n🗑️ Removing: {model_name}")

    run_cmd([
        "ollama",
        "rm",
        model_name
    ])

    with OUTPUT:

        print("✅ Deleted!")

# ==============================================================================
# INFO
# ==============================================================================
def show_model_info(model_name):

    detail = STATE["details"].get(model_name)

    if not detail:
        detail = fetch_model_detail(model_name)

    with OUTPUT:

        print("\n" + "═" * 120)

        print(f"MODEL: {model_name}")

        print("═" * 120)

        print(f"SIZE: {detail.get('size')}")
        print(f"CONTEXT: {detail.get('context')}")
        print(f"DESC: {detail.get('description')}")

# ==============================================================================
# RENDER
# ==============================================================================
def render_catalog():

    with OUTPUT:

        clear_output(wait=True)

        print("═" * 150)

        print(
            "ID   | STATUS         | MODEL                        | SIZE      | CTX       | DESCRIPTION"
        )

        print("─" * 150)

        installed = set(
            STATE["installed"]
        )

        for i, slug in enumerate(
            STATE["library"],
            1
        ):

            status = (
                "✅ INSTALLED"
                if slug in installed
                else "⚪ EMPTY"
            )

            detail = STATE["details"].get(
                slug,
                {}
            )

            fallback = MODEL_DB.get(
                slug.lower(),
                {}
            )

            size = (
                detail.get("size")
                or fallback.get("s", "---")
            )

            ctx = (
                detail.get("context")
                or fallback.get("c", "---")
            )

            desc = (
                detail.get("description")
                or fallback.get("n", "")
            )

            if len(desc) > 70:
                desc = desc[:70] + "..."

            print(
                f"{str(i):<4} | "
                f"{status:<14} | "
                f"{slug:<28} | "
                f"{size:<9} | "
                f"{ctx:<9} | "
                f"{desc}"
            )

        print("\n📂 INSTALLED MODELS:\n")

        for i, name in enumerate(
            STATE["installed"],
            101
        ):

            print(
                f"{i:<4} | 🗑️ {name}"
            )

        print("\n═" * 150)

        print(f"📦 TOTAL MODELS: {len(STATE['library'])}")

        print("\nCOMMANDS:")
        print("1                -> install")
        print("delete 101       -> delete")
        print("info 1           -> info")
        print("refresh          -> refresh")
        print("stop ngrok       -> stop ngrok")
        print("stop server      -> stop ollama")
        print("0                -> start ngrok")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================
def process_command(choice):

    low = choice.lower()

    if low == "0":

        run_ngrok()
        return

    if low == "refresh":

        refresh_all_data()
        render_catalog()
        return

    if low == "stop ngrok":

        stop_ngrok()
        return

    if low == "stop server":

        stop_ollama_server()
        return

    if low.startswith("info "):

        arg = choice.split(" ", 1)[1]

        if arg.isdigit():

            idx = int(arg) - 1

            if 0 <= idx < len(STATE["library"]):

                show_model_info(
                    STATE["library"][idx]
                )

        else:

            show_model_info(arg)

        return

    if low.startswith("delete "):

        try:

            delete_id = int(
                choice.split()[1]
            )

            idx = delete_id - 101

            if 0 <= idx < len(STATE["installed"]):

                target = STATE["installed"][idx]

                delete_model(target)

                refresh_all_data()
                render_catalog()

        except:

            with OUTPUT:

                print("⚠️ DELETE FAILED")

        return

    if choice.isdigit():

        idx = int(choice) - 1

        if 0 <= idx < len(STATE["library"]):

            target = STATE["library"][idx]

            pull_model(target)

            refresh_all_data()
            render_catalog()

        return

    pull_model(choice)

    refresh_all_data()
    render_catalog()

# ==============================================================================
# SUBMIT
# ==============================================================================
def on_submit(_=None):

    cmd = INPUT_BOX.value.strip()

    INPUT_BOX.value = ""

    if not cmd:
        return

    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================
restart_ollama_server()

refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================
TITLE = widgets.HTML("""
<h2>🚀 Ollama Colab Manager</h2>
""")

HELP = widgets.HTML("""
<b>
Commands:
1 | delete 101 | info 1 | refresh
</b>
""")

INPUT_BOX = widgets.Text(
    placeholder="Nhập command...",
    description=">>>",
    layout=widgets.Layout(width="90%")
)

SEND_BTN = widgets.Button(
    description="Send",
    button_style="success"
)

REFRESH_BTN = widgets.Button(
    description="Refresh",
    button_style="info"
)

NGROK_BTN = widgets.Button(
    description="Start Ngrok",
    button_style="warning"
)

STOP_NGROK_BTN = widgets.Button(
    description="Stop Ngrok",
    button_style="danger"
)

STOP_SERVER_BTN = widgets.Button(
    description="Stop Server",
    button_style="danger"
)

# ==============================================================================
# EVENTS
# ==============================================================================
SEND_BTN.on_click(on_submit)

INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(
    lambda _: (
        refresh_all_data(),
        render_catalog()
    )
)

NGROK_BTN.on_click(
    lambda _: run_ngrok()
)

STOP_NGROK_BTN.on_click(
    lambda _: stop_ngrok()
)

STOP_SERVER_BTN.on_click(
    lambda _: stop_ollama_server()
)

# ==============================================================================
# BUTTONS
# ==============================================================================
BUTTONS = widgets.HBox([

    SEND_BTN,
    REFRESH_BTN,
    NGROK_BTN,
    STOP_NGROK_BTN,
    STOP_SERVER_BTN

])

# ==============================================================================
# MAIN UI
# ==============================================================================
UI = widgets.VBox([

    TITLE,

    OUTPUT,

    HELP,

    INPUT_BOX,

    BUTTONS

])

display(UI)

render_catalog()

KeyboardInterrupt: 

claude product


In [ ]:
# ==============================================================================
# OLLAMA COLAB MANAGER - ULTIMATE PATCHED FULL SRC
#
# FEATURES
# ==============================================================================
# ✅ Ollama Library Crawl
# ✅ HuggingFace GGUF Crawl  (FIXED: pagination, context, size)
# ✅ LM Studio Crawl          (FIXED: pagination, context, size)
# ✅ Provider Sort
# ✅ Pagination
# ✅ Page Jump
# ✅ Delete by ID
# ✅ Delete by Name
# ✅ Info Command
# ✅ Info shows URL
# ✅ Context Detection        (FIXED: reads config.json)
# ✅ Quant Detection          (FIXED: best-first priority order)
# ✅ Compatibility Scoring
# ✅ URL Metadata
# ✅ Realtime Pull Logs
# ✅ Ngrok
# ✅ Stop Server
# ✅ Stop Ngrok
# ✅ Colab Widgets UI
# ✅ Keeps Original UI Logic
# ==============================================================================

import os
import re
import time
import math
import requests
import subprocess

from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote, unquote

import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# CONFIG
# ==============================================================================

OLLAMA_HOST     = "0.0.0.0:11434"
OLLAMA_PORT     = "11434"

MODELS_PER_PAGE = 35
CURRENT_PAGE    = 0

MAX_LIBRARY_PAGES = 25
HTTP_TIMEOUT      = 30

HF_MAX_MODELS     = 1200
# HF_PAGE_LIMIT     = 100   # real HF API max per page

LMSTUDIO_PUBLISHERS = [
    "lmstudio-community",
    "bartowski",
    "QuantFactory",
    "unsloth",
]

# ==============================================================================
# STATE
# ==============================================================================

STATE = {
    "installed": [],
    "library":   [],
    "details":   {},
}

# ==============================================================================
# MODEL DB
# ==============================================================================

MODEL_DB = {
    "llama3.1":      {"s": "4.7GB",  "c": "128K", "n": "General assistant, multilingual, strong Vietnamese support."},
    "llama3.2":      {"s": "2.0GB",  "c": "128K", "n": "Lightweight and fast."},
    "deepseek-r1":   {"s": "4.7GB+", "c": "128K", "n": "Excellent reasoning and coding."},
    "qwen2.5-coder": {"s": "4.7GB",  "c": "128K", "n": "Coding optimized."},
    "phi4":          {"s": "8GB",    "c": "128K", "n": "Strong logic and reasoning."},
    "gemma2":        {"s": "5.5GB",  "c": "8K",   "n": "Google model."},
    "mistral":       {"s": "4.1GB",  "c": "32K",  "n": "Stable chatbot."},
}

# ==============================================================================
# QUANT PRIORITY  (best → worst — first match wins)
# ==============================================================================

QUANT_PRIORITY = [
    "q8_0",
    "q6_k",
    "q5_k_m", "q5_k_s",
    "q4_k_m", "q4_k_s", "q4_0",
    "q3_k_m", "q3_k_s", "q3_k_l",
    "q2_k",
]

# Keys to look for in config.json when reading context window size
HF_CONTEXT_KEYS = [
    "max_position_embeddings",   # most common (llama, mistral, qwen …)
    "n_positions",               # GPT-2 style
    "max_seq_len",
    "model_max_length",
    "seq_length",
    "sliding_window",
    "max_sequence_length",
]

# ==============================================================================
# CMD
# ==============================================================================

def run_cmd(args, timeout=120):
    try:
        p = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        return p.returncode, p.stdout.strip(), p.stderr.strip()
    except Exception as e:
        return 1, "", str(e)

# ==============================================================================
# CURL
# ==============================================================================

def curl_get(url):
    code, out, err = run_cmd([
        "curl", "-L", "-sS",
        "--max-time", str(HTTP_TIMEOUT),
        url,
    ])
    return out if code == 0 else ""

# ==============================================================================
# SIZE FORMAT
# ==============================================================================

def format_bytes(size):
    try:
        size = int(size)
    except Exception:
        return "---"
    if size <= 0:
        return "---"
    gb = size / (1024 ** 3)
    if gb >= 1:
        return f"{gb:.1f}GB"
    mb = size / (1024 ** 2)
    return f"{mb:.0f}MB"

# ==============================================================================
# PROVIDER SORT
# ==============================================================================

def get_provider_priority(name):
    if name.startswith("lmstudio/"):
        return 1
    if name.startswith("hf.co/"):
        return 2
    return 0   # official Ollama first

# ==============================================================================
# COMPAT SCORE
# ==============================================================================

def evaluate_compatibility(model_name, gguf_files):
    score   = 0
    reasons = []
    lower   = [f.lower() for f in gguf_files]

    if gguf_files:
        score += 35
        reasons.append("GGUF")

    quant_keywords = ["q4_k_m", "q5_k_m", "q8_0", "q4_0", "q6_k"]
    if any(q in f for q in quant_keywords for f in lower):
        score += 30
        reasons.append("QUANT")

    if any("instruct" in f or "chat" in f for f in lower):
        score += 15
        reasons.append("CHAT")

    if any("gguf" in f for f in lower):
        score += 20
        reasons.append("OLLAMA_OK")

    return score >= 60, score, ", ".join(reasons)

# ==============================================================================
# QUANT HELPER  (best-first)
# ==============================================================================

def best_quant(gguf_files):
    """Return highest-quality quant found — best first in QUANT_PRIORITY."""
    lower = [f.lower() for f in gguf_files]
    for q in QUANT_PRIORITY:
        if any(q in f for f in lower):
            return q.upper()
    return "---"

# ==============================================================================
# HF PAGINATION  (follows Link: rel="next" header)
# ==============================================================================

def hf_paginate(url_base, max_models=HF_MAX_MODELS):
    """
    Walk all pages of a HuggingFace API endpoint using the Link header.
    The HF API silently caps ?limit= at 100 per page; this function
    keeps following 'next' links until max_models are collected or
    there are no more pages.
    """
    collected = []
    url = url_base

    while url and len(collected) < max_models:
        try:
            resp = requests.get(url, timeout=40)
            resp.raise_for_status()
            data = resp.json()

            if not data:
                break

            collected.extend(data)

            # Parse  Link: <url>; rel="next", <url>; rel="last"
            next_url  = None
            link_hdr  = resp.headers.get("Link", "")
            for part in link_hdr.split(","):
                part = part.strip()
                if 'rel="next"' in part:
                    m = re.search(r"<([^>]+)>", part)
                    if m:
                        next_url = m.group(1)
                        break
            url = next_url

        except Exception as e:
            print(f"HF PAGINATE ERROR: {e}")
            break

    return collected[:max_models]

# ==============================================================================
# HF CONTEXT  (reads config.json per model)
# ==============================================================================

def fetch_hf_context(model_id):
    """
    Fetch context window size from the model's config.json.
    Returns a human-readable string like '128K', '1M', or '---'.
    """
    try:
        url  = f"https://huggingface.co/{model_id}/resolve/main/config.json"
        resp = requests.get(url, timeout=15)
        if resp.status_code != 200:
            return "---"

        cfg = resp.json()

        for key in HF_CONTEXT_KEYS:
            val = cfg.get(key)
            if isinstance(val, int) and val > 0:
                if val >= 1_000_000:
                    return f"{val // 1_000_000}M"
                elif val >= 1_000:
                    return f"{val // 1_000}K"
                else:
                    return str(val)

    except Exception:
        pass

    return "---"

# ==============================================================================
# HF BEST GGUF SIZE  (single best-quant file, not total repo size)
# ==============================================================================

def hf_best_gguf_size(siblings):
    """
    Return (size_bytes, gguf_file_list).
    Size is taken from the single best-quant GGUF file rather than
    summing every GGUF in the repo (which inflates the number).
    Falls back to the largest GGUF if preferred quant not found.
    """
    gguf = [
        s for s in siblings
        if s.get("rfilename", "").lower().endswith(".gguf")
    ]
    if not gguf:
        return 0, []

    gguf_files = [s["rfilename"] for s in gguf]

    # Try each quant in best-first order
    for q in QUANT_PRIORITY:
        for s in gguf:
            if q in s["rfilename"].lower():
                return s.get("size") or 0, gguf_files

    # Fallback: pick the largest file
    best = max(gguf, key=lambda s: s.get("size") or 0)
    return best.get("size") or 0, gguf_files

# ==============================================================================
# BUILD MODEL ENTRY  (shared by HF + LM Studio)
# ==============================================================================

def build_hf_entry(item, prefix, fetch_context=True):
    """
    Convert a raw HF API model dict into a detail dict.
    prefix is either 'hf.co' or 'lmstudio'.
    """
    model_id = item.get("id", "")
    if not model_id:
        return None

    siblings           = item.get("siblings", [])
    size_bytes, gguf_files = hf_best_gguf_size(siblings)

    if not gguf_files:
        return None

    compat, score, reason = evaluate_compatibility(model_id, gguf_files)
    quant   = best_quant(gguf_files)
    context = fetch_hf_context(model_id) if fetch_context else "---"

    # Try model card summary first, then fallback
    desc = ""
    card = item.get("cardData") or {}
    desc = card.get("summary", "") or item.get("description", "") or ""

    return {
        "name":        f"{prefix}/{model_id}",
        "quant":       quant,
        "compatible":  compat,
        "score":       score,
        "reason":      reason,
        "size":        format_bytes(size_bytes),
        "context":     context,
        "description": desc[:120],
        "url":         f"https://huggingface.co/{model_id}",
    }

# ==============================================================================
# FETCH HF  (FIXED)
# ==============================================================================

def fetch_huggingface_models():
    models = []

    try:
        url_base = (
            "https://huggingface.co/api/models"
            "?search=GGUF"
            # f"&limit={HF_PAGE_LIMIT}"
            "&sort=downloads"   # most popular first
            "&full=true"
        )

        print(f"🤗 Crawling HuggingFace (up to {HF_MAX_MODELS} models)…")
        raw = hf_paginate(url_base, max_models=HF_MAX_MODELS)
        print(f"🤗 Found {len(raw)} raw entries — fetching context sizes…")

        # Fetch config.json in parallel to keep things fast
        def _build(item):
            return build_hf_entry(item, prefix="hf.co", fetch_context=True)

        with ThreadPoolExecutor(max_workers=16) as pool:
            futures = {pool.submit(_build, item): item for item in raw}
            for future in as_completed(futures):
                result = future.result()
                if result:
                    models.append(result)

    except Exception as e:
        print(f"HF ERROR: {e}")

    print(f"🤗 HuggingFace: {len(models)} GGUF models loaded.")
    return models

# ==============================================================================
# FETCH LM STUDIO  (FIXED)
# ==============================================================================

def fetch_lmstudio_models():
    models = []

    for publisher in LMSTUDIO_PUBLISHERS:
        try:
            url_base = (
                f"https://huggingface.co/api/models"
                f"?author={publisher}"
                # f"&limit={HF_PAGE_LIMIT}"
                f"&sort=downloads"
                f"&full=true"
            )

            print(f"🧠 Crawling LM Studio publisher: {publisher}…")
            raw = hf_paginate(url_base, max_models=500)
            print(f"🧠 {publisher}: {len(raw)} raw entries — fetching context sizes…")

            def _build(item):
                return build_hf_entry(item, prefix="lmstudio", fetch_context=True)

            with ThreadPoolExecutor(max_workers=16) as pool:
                futures = {pool.submit(_build, item): item for item in raw}
                for future in as_completed(futures):
                    result = future.result()
                    if result:
                        models.append(result)

        except Exception as e:
            print(f"LM ERROR ({publisher}): {e}")

    print(f"🧠 LM Studio: {len(models)} GGUF models loaded.")
    return models

# ==============================================================================
# RESTART SERVER
# ==============================================================================

def restart_ollama_server():
    print("🔄 Starting Ollama…")
    run_cmd(["bash", "-lc", "pkill ollama || true"])
    os.environ["OLLAMA_HOST"] = OLLAMA_HOST
    log_file = open("ollama.log", "w")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=log_file,
    )
    time.sleep(6)
    print("✅ Ollama Ready!")

# ==============================================================================
# STOP SERVER
# ==============================================================================

def stop_ollama_server():
    print("\n🛑 Stopping Ollama…")
    run_cmd(["bash", "-lc", "pkill ollama || true"])
    print("✅ Ollama Stopped!")

# ==============================================================================
# NGROK
# ==============================================================================

def run_ngrok():
    print("\n🚀 Starting Ngrok…")
    run_cmd(["bash", "-lc", f"nohup ngrok http {OLLAMA_PORT} > /dev/null 2>&1 &"])
    time.sleep(3)
    try:
        response = requests.get("http://localhost:4040/api/tunnels", timeout=5)
        tunnels  = response.json().get("tunnels", [])
        if tunnels:
            url = tunnels[0]["public_url"]
            print(f"\n✅ NGROK ONLINE:\n{url}")
        else:
            print("⚠️ No Tunnel Found")
    except Exception:
        print("⚠️ Ngrok Error")

# ==============================================================================
# STOP NGROK
# ==============================================================================

def stop_ngrok():
    print("\n🛑 Stopping Ngrok…")
    run_cmd(["bash", "-lc", "pkill ngrok || true"])
    print("✅ Ngrok Stopped!")

# ==============================================================================
# INSTALLED MODELS
# ==============================================================================

def get_installed_models():
    code, out, err = run_cmd(["ollama", "list"], timeout=20)
    if code != 0:
        return []
    lines = out.splitlines()
    if len(lines) <= 1:
        return []
    models = []
    for line in lines[1:]:
        parts = line.split()
        if not parts:
            continue
        models.append(parts[0].split(":")[0])
    return models

# ==============================================================================
# FETCH OLLAMA LIBRARY
# ==============================================================================

def fetch_library_models():
    models = []
    for page in range(1, MAX_LIBRARY_PAGES + 1):
        try:
            url  = "https://ollama.com/library" if page == 1 else f"https://ollama.com/library?page={page}"
            html = curl_get(url)
            if not html:
                continue
            found = re.findall(r'href="/library/([^"]+)"', html)
            for item in found:
                item = unquote(item)
                if "/" in item:
                    continue
                if item not in models:
                    models.append(item)
        except Exception:
            pass
    return models

# ==============================================================================
# FETCH OLLAMA DETAILS
# ==============================================================================

def fetch_ollama_details(slug):
    result = {
        "size":        "---",
        "context":     "---",
        "description": "",
        "url":         f"https://ollama.com/library/{slug}",
        "compat":      True,
        "score":       100,
        "reason":      "Official Ollama",
        "quant":       "---",
    }
    try:
        html = curl_get(f"https://ollama.com/library/{quote(slug)}")
        if html:
            desc = re.search(r'<meta name="description" content="([^"]+)"', html)
            if desc:
                result["description"] = desc.group(1)

        tags_html = curl_get(f"https://ollama.com/library/{quote(slug)}/tags")
        if tags_html:
            text = re.sub(r"<[^>]+>", " ", tags_html)
            size_match = re.search(r"([0-9.]+\s?(?:GB|MB))", text)
            ctx_match  = re.search(r"([0-9.]+\s?[KMG]?)\s*context", text, flags=re.I)
            if size_match:
                result["size"]    = size_match.group(1)
            if ctx_match:
                result["context"] = ctx_match.group(1)
    except Exception:
        pass
    return result

# ==============================================================================
# REFRESH
# ==============================================================================

def refresh_all_data():
    STATE["installed"] = get_installed_models()

    ollama_models = fetch_library_models()
    hf_models     = fetch_huggingface_models()
    lm_models     = fetch_lmstudio_models()

    merged = list(ollama_models)
    merged.extend(x["name"] for x in hf_models)
    merged.extend(x["name"] for x in lm_models)
    merged = list(dict.fromkeys(merged))   # deduplicate, preserve order

    merged = sorted(
        merged,
        key=lambda x: (get_provider_priority(x), x.lower()),
    )

    STATE["library"] = merged
    STATE["details"] = {}

    # Ollama details
    for slug in ollama_models:
        STATE["details"][slug] = fetch_ollama_details(slug)

    # HF + LM Studio details
    for item in hf_models + lm_models:
        STATE["details"][item["name"]] = {
            "compat":      item["compatible"],
            "score":       item["score"],
            "reason":      item["reason"],
            "quant":       item["quant"],
            "size":        item["size"],
            "context":     item["context"],
            "description": item["description"],
            "url":         item["url"],
        }

# ==============================================================================
# PULL
# ==============================================================================

def pull_model(model_name):
    print("\n" + "═" * 100)
    print(f"📥 DOWNLOADING: {model_name}")
    print("═" * 100)

    try:
        process = subprocess.Popen(
            ["ollama", "pull", model_name],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        last_line = ""
        while True:
            line = process.stdout.readline()
            if not line and process.poll() is not None:
                break
            if not line:
                continue
            line = line.strip()
            if not line:
                continue

            pretty = line
            pretty = pretty.replace("pulling manifest",      "📦 pulling manifest")
            pretty = pretty.replace("verifying sha256 digest", "🔐 verifying digest")
            pretty = pretty.replace("writing manifest",      "💾 writing manifest")
            pretty = pretty.replace("success",               "✅ success")
            if "%" in pretty:
                pretty = "⬇️  " + pretty

            if pretty != last_line:
                print(pretty)
                last_line = pretty

        rc = process.poll()
        print("\n" + "═" * 100)
        print("✅ DOWNLOAD SUCCESS" if rc == 0 else "⚠️ DOWNLOAD FAILED")

    except Exception as e:
        print(f"❌ ERROR: {e}")

# ==============================================================================
# DELETE
# ==============================================================================

def delete_model(model_name):
    print(f"\n🗑️  Removing: {model_name}")
    run_cmd(["ollama", "rm", model_name])
    print("✅ Removed!")

# ==============================================================================
# INFO
# ==============================================================================

def show_model_info(model_name):
    detail   = STATE["details"].get(model_name, {})
    model_id = None
    try:
        model_id = STATE["library"].index(model_name) + 1
    except Exception:
        pass

    print("\n" + "═" * 100)
    print(f"MODEL: {model_name}")
    if model_id:
        print(f"ID:    {model_id}")
    print("═" * 100)
    print(f"SIZE:       {detail.get('size',        '---')}")
    print(f"CONTEXT:    {detail.get('context',     '---')}")
    print(f"QUANT:      {detail.get('quant',       '---')}")
    print(f"COMPATIBLE: {detail.get('compat',      '---')}")
    print(f"SCORE:      {detail.get('score',       '---')}/100")
    print(f"REASON:     {detail.get('reason',      '---')}")
    print(f"DESC:       {detail.get('description', '---')}")
    print(f"URL:        {detail.get('url',         '---')}")

# ==============================================================================
# PROCESS COMMAND
# ==============================================================================

def process_command(choice):
    global CURRENT_PAGE

    low = choice.lower().strip()

    # page <n>
    if low.startswith("page "):
        try:
            page_num = int(low.split()[1])
            max_page = max(1, math.ceil(len(STATE["library"]) / MODELS_PER_PAGE))
            if 1 <= page_num <= max_page:
                CURRENT_PAGE = page_num - 1
                render_catalog()
        except Exception:
            print("⚠️ PAGE ERROR")
        return

    # delete "name"
    if low.startswith('delete "'):
        try:
            model_name = re.findall(r'delete\s+"(.+)"', choice, flags=re.I)[0]
            delete_model(model_name)
            refresh_all_data()
            render_catalog()
        except Exception:
            print("⚠️ DELETE ERROR")
        return

    # delete <id>
    if low.startswith("delete "):
        try:
            delete_id = int(choice.split()[1])
            idx = delete_id - 101
            if 0 <= idx < len(STATE["installed"]):
                delete_model(STATE["installed"][idx])
                refresh_all_data()
                render_catalog()
        except Exception:
            print("⚠️ DELETE ERROR")
        return

    # info <id|name>
    if low.startswith("info "):
        arg = choice.split(" ", 1)[1]
        if arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(STATE["library"]):
                show_model_info(STATE["library"][idx])
        else:
            show_model_info(arg)
        return

    if low == "refresh":
        refresh_all_data()
        render_catalog()
        return

    if low == "stop ngrok":
        stop_ngrok()
        return

    if low == "stop server":
        stop_ollama_server()
        return

    if low == "0":
        run_ngrok()
        return

    # numeric → pull by catalog ID
    if choice.isdigit():
        idx = int(choice) - 1
        if 0 <= idx < len(STATE["library"]):
            pull_model(STATE["library"][idx])
            refresh_all_data()
            render_catalog()
        return

    # plain text → treat as model name / pull URL
    pull_model(choice)
    refresh_all_data()
    render_catalog()

# ==============================================================================
# RENDER
# ==============================================================================

def render_catalog():
    with OUTPUT:
        clear_output(wait=True)

        total_models = len(STATE["library"])
        max_page     = max(1, math.ceil(total_models / MODELS_PER_PAGE))
        start        = CURRENT_PAGE * MODELS_PER_PAGE
        end          = start + MODELS_PER_PAGE
        page_models  = STATE["library"][start:end]
        installed    = set(STATE["installed"])

        print("═" * 190)
        print(
            "ID   | SRC  | STATUS         | MODEL                              "
            "| SIZE     | CTX     | QUANT    | COMPAT | DESCRIPTION"
        )
        print("─" * 190)

        for offset, slug in enumerate(page_models):
            i      = start + offset + 1
            detail = STATE["details"].get(slug, {})

            compat = detail.get("compat")
            quant  = detail.get("quant",   "---")
            ctx    = detail.get("context", "---")
            size   = detail.get("size",    "---")

            if slug.startswith("hf.co/"):
                source = "🤗🟢" if compat is True else ("🤗🔴" if compat is False else "🤗")
            elif slug.startswith("lmstudio/"):
                source = "🧠🟢" if compat is True else ("🧠🔴" if compat is False else "🧠")
            else:
                source = "🦙"

            status       = "✅ INSTALLED" if slug in installed else "⚪ EMPTY"
            desc         = detail.get("description", "")
            if len(desc) > 55:
                desc = desc[:55] + "..."
            compat_text  = str(detail.get("score", "---"))
            display_slug = slug[:34]

            print(
                f"{str(i):<4} | "
                f"{source:<5} | "
                f"{status:<14} | "
                f"{display_slug:<34} | "
                f"{size:<8} | "
                f"{ctx:<7} | "
                f"{quant:<8} | "
                f"{compat_text:<7} | "
                f"{desc}"
            )

        print("\n📂 INSTALLED MODELS:")
        for i, name in enumerate(STATE["installed"], 101):
            print(f"{i:<4} | 🗑️  {name}")

        print("\n" + "═" * 190)

        ollama_count   = sum(1 for x in STATE["library"] if not x.startswith(("hf.co/", "lmstudio/")))
        lmstudio_count = sum(1 for x in STATE["library"] if x.startswith("lmstudio/"))
        hf_count       = sum(1 for x in STATE["library"] if x.startswith("hf.co/"))

        print(f"🦙 Ollama: {ollama_count} | 🧠 LM Studio: {lmstudio_count} | 🤗 HF: {hf_count}")
        print(f"\n📄 PAGE: {CURRENT_PAGE + 1}/{max_page}")

        print("\nCOMMANDS:")
        print("1                    -> pull model by catalog ID")
        print("delete 101           -> delete installed model by ID")
        print('delete "llama3"      -> delete installed model by name')
        print("info 1               -> show model info")
        print("page 5               -> jump to page")
        print("refresh              -> refresh all data")
        print("stop ngrok           -> stop ngrok")
        print("stop server          -> stop ollama server")
        print("0                    -> start ngrok")

# ==============================================================================
# SUBMIT
# ==============================================================================

def on_submit(_=None):
    cmd = INPUT_BOX.value.strip()
    INPUT_BOX.value = ""
    if not cmd:
        return
    with OUTPUT:
        print(f"\n👉 {cmd}")
    process_command(cmd)

# ==============================================================================
# START
# ==============================================================================

restart_ollama_server()
refresh_all_data()

# ==============================================================================
# UI
# ==============================================================================

TITLE = widgets.HTML("<h2>🚀 Ollama Colab Manager</h2>")

HELP = widgets.HTML("""
<b>Commands: 1 | delete 101 | delete "name" | info 1 | page 5 | refresh | stop ngrok | stop server | 0</b>
""")

INPUT_BOX = widgets.Text(
    placeholder="Nhập lệnh...",
    description=">>>",
    layout=widgets.Layout(width="90%"),
)

SEND_BTN      = widgets.Button(description="Gửi",        button_style="success")
REFRESH_BTN   = widgets.Button(description="Refresh",    button_style="info")
NGROK_BTN     = widgets.Button(description="Start Ngrok", button_style="warning")
STOP_NGROK_BTN  = widgets.Button(description="Stop Ngrok",   button_style="danger")
STOP_SERVER_BTN = widgets.Button(description="Stop Server",  button_style="danger")

OUTPUT = widgets.Output()

# ==============================================================================
# EVENTS
# ==============================================================================

SEND_BTN.on_click(on_submit)
INPUT_BOX.on_submit(on_submit)

REFRESH_BTN.on_click(lambda _: (refresh_all_data(), render_catalog()))
NGROK_BTN.on_click(lambda _: run_ngrok())
STOP_NGROK_BTN.on_click(lambda _: stop_ngrok())
STOP_SERVER_BTN.on_click(lambda _: stop_ollama_server())

# ==============================================================================
# LAYOUT
# ==============================================================================

BUTTONS = widgets.HBox([
    SEND_BTN,
    REFRESH_BTN,
    NGROK_BTN,
    STOP_NGROK_BTN,
    STOP_SERVER_BTN,
])

UI = widgets.VBox([
    TITLE,
    OUTPUT,
    HELP,
    INPUT_BOX,
    BUTTONS,
])

display(UI)
render_catalog()